# Passive Voice Detector - Test Set Evaluation



## STEP 1: Install & Import Libraries

In [ ]:
import pandas as pd
import re
import spacy
from spacy.matcher import Matcher
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
# Make pandas show full text in cells
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

print('✅ All libraries loaded successfully.')


## STEP 2: Load the Labeled Test Set


In [ ]:
df = pd.read_csv('/content/group1.csv', sep=None, engine='python', on_bad_lines='warn')

print(f'Columns detected after reading CSV: {df.columns.tolist()}')

# The original CSV columns are: ['text', 'Student number:', 'Ln'].
# Correctly assign them to DataFrame columns 'text', 'student_number', and 'correct'.
# Assuming 'Ln' corresponds to the 'correct' passive voice counts.
# If a separate 'error' column exists in your data, you'll need to adjust this.
df.columns = ['text', 'student_number', 'correct']

# Ensure the 'error' column exists, initialized to 0 as it's a human-annotated count not present in group1.csv.
# This ensures the DataFrame has the 'error' column needed for later steps.
if 'error' not in df.columns:
    df['error'] = 0

# Convert 'correct' column to numeric, replacing 'x' with 0, as 'x' values were observed before.
df['correct'] = df['correct'].replace('x', 0).astype(int)

print(f'✅ Loaded {len(df)} rows.')
print(f'   Columns: {df.columns.tolist()}')
print()
display(df.head(10))

---
## STEP 3: Clean the Text


In [ ]:
import pandas as pd
import re

# --- STEP 1: Load and Rename Columns ---
df = pd.read_csv('/content/group1.csv', sep=None, engine='python', on_bad_lines='warn')

print(f'Columns detected after reading CSV: {df.columns.tolist()}')

# Manually assign columns: 'text', 'student_number', 'correct'
df.columns = ['text', 'student_number', 'correct']

# Ensure 'error' column exists
if 'error' not in df.columns:
    df['error'] = 0

# Clean 'correct' column
df['correct'] = df['correct'].replace('x', 0).astype(int)

# --- STEP 2: Define Cleaning & Splitting Functions ---
def clean_spaces(text):
    if isinstance(text, str):
        # Remove multiple spaces/tabs/newlines
        text = re.sub(r'\s+', ' ', text)
        return text.strip()
    return text

def split_into_sentences(text):
    if isinstance(text, str):
        # Split on punctuation followed by whitespace
        sentences = re.split(r'(?<=[.!?])\s+', text)
        return [s.strip() for s in sentences if s.strip()]
    return []

# --- STEP 3: Process and Flatten to Sentence Level ---
# 1. Clean the paragraph text
df['text'] = df['text'].apply(clean_spaces)

# 2. Convert each paragraph into a list of sentences
df['text'] = df['text'].apply(split_into_sentences)

# 3. "Explode" the lists into individual rows (the column name remains 'text')
df = df.explode('text').reset_index(drop=True)

# 4. Remove any nulls or empty strings resulting from the split
df = df.dropna(subset=['text'])
df = df[df['text'].str.strip() != ''].reset_index(drop=True)

print(f'✅ Process Complete.')
print(f'   Total sentences in "text" column: {len(df)}')
print()
display(df[['student_number', 'text', 'correct', 'error']].head(15))

#STEP 4: CLASSIFY THE TEXTS USING RULE-BASED

In [ ]:
def correct(new_text):
    # Parse the input text
    doc = nlp(new_text)
    matcher = Matcher(nlp.vocab)

    # Define patterns for passive voice detection (Correct usage)
    patterns = [
        #1 be + VBN (Singular) (Pronouns, name, and sing noun)
         [
            {
                "TAG": "PRP", "DEP": "nsubjpass",
                "LOWER": {"IN": ["he", "she", "it"]}
            },
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBZ"]}
            },
            {
                "TAG": {"IN": ["RB", "RBS"]},
                "OP": "?"
            },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
            },
            {"TAG": "VBN", "DEP": "ROOT"},
             {
                "POS": "ADV",
                "OP": "*"
            },
            {
                "DEP": "agent",
                "LOWER": "by"
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "pobj"
            }
        ],
        [
            {
                "TAG": "PRP",
                "LOWER": {"IN": ["he", "she", "it"]},
                "DEP": {"IN": ["nsubj", "nsubjpass"]}
            },
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBZ"]},
                "DEP": "auxpass"
            },
            {"TAG": "VBN",
             "DEP": "ROOT"},
             {
                "POS": "ADV"}
        ],

[
    {"LOWER": "it"},
    {"LOWER": "'s"},
    {"TAG": "VBN", "DEP": "ROOT"},
    {
                "DEP": "oprd"}

],
         [
            {
                "TAG": "PRP",
                "LOWER": {"IN": ["he", "she", "it"]},
                "DEP": {"IN": ["nsubj", "nsubjpass"]}
            },
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBZ"]},
                "DEP": "auxpass"
            },
            {"TAG": "RB", "OP": "?"},
            {"TAG": "VBN",
             "DEP": "ROOT"}
        ],
         [
            {
                "TAG": "PRP",
                "LEMMA": {"IN": ["he", "she", "it"]},
            },
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBZ"]},
                "DEP": "auxpass"
            },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
            },
            {"TAG": "VBN",
             "DEP": "ROOT"}
        ],
        [
            {
                "TAG": "PRP",
                "LOWER": {"IN": ["he", "she", "it"]},
            },
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBZ"]}
            },
            {
                "TAG": {"IN": ["RB", "RBS"]},
                "OP": "?"
            },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
            },
            {"TAG": "VBN"}
        ],
        [
            {

                "TAG": {"IN": ["NNP", "NN"]}
            },
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "aux", "mark", "punct"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["RB", "RBR"]},
                "OP": "?"
            },
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "acomp"]},
                "OP": "*"
            },
             {
                "LEMMA": "be",
                "TAG": "VBZ", "TEXT": "is"
             },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"},
           {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "acomp"]},
                "OP": "*"
            },
            {"TAG": "VBN"},
           {
                "DEP": "agent",
                "LOWER": "by"
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "prep"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "pobj"
            }

            ],
        [
            {

                "TAG": {"IN": ["NNP", "NN"]}
            },
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "aux", "mark", "punct"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["RB", "RBR"]},
                "OP": "?"
            },
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "acomp"]},
                "OP": "*"
            },
             {
                "LEMMA": "be",
                "TAG": "VBD", "TEXT": "was"
             },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"},
           {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "acomp"]},
                "OP": "*"
            },
            {"TAG": "VBN"},
           {
                "DEP": "agent",
                "LOWER": "by"
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "prep"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "pobj"
            }

            ],
        [
            {

                "TAG": {"IN": ["NNP", "NN"]}
            },
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "aux", "mark", "punct"]},
                "OP": "*"
            },
           {"DEP": "nsubj"},
           {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "aux", "mark", "punct", "ccomp", "acomp"]},
                "OP": "*"
            },
             {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBZ"]}
             },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"},
            {"TAG": "VBN"}

            ],
         [
            {

                "TAG": {"IN": ["NNP", "NN"]}
            },
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "aux", "mark", "punct"]},
                "OP": "*"
            },
           {"DEP": "nsubj"},
           {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "aux", "mark", "punct", "ccomp", "acomp"]},
                "OP": "*"
            },
             {
                "LEMMA": "be",
                "TAG": "VBD", "TEXT": "was"
             },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"},
            {"TAG": "VBN"}

            ],
        [
            {

                "TAG": {"IN": ["NNP", "NN"]}, "DEP": "nsubjpass"
            },
             {
                "LEMMA": "be",
                "TAG": "VBZ", "TEXT": "is"
             },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"},
            {"TAG": "VBN"}
            ],
        [
            {

                "TAG": {"IN": ["NNP", "NN"]}, "DEP": "nsubjpass"
            },
             {
                "LEMMA": "be",
                "TAG": "VBD", "TEXT": "was"
             },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"},
            {"TAG": "VBN"},
           {
                "DEP": "agent",
                "LOWER": "by"
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "pobj"
            }

            ],
         [
            {

                "TAG": {"IN": ["NNP", "NN"]}, "DEP": "nsubjpass"
            },
             {
                "LEMMA": "be",
                "TAG": "VBZ", "TEXT": "is"
             },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"},
            {"TAG": "VBN"},
           {
                "DEP": "agent",
                "LOWER": "by"
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "pobj"
            }

            ],
        [

            {

                "TAG": {"IN": ["NNP", "NN"]}, "DEP": "nsubjpass"
            },
             {
                "LEMMA": "be",
                "TAG": "VBD", "TEXT": "was"
             },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"},
            {"TAG": "VBN"}

            ],
        [

            {

                "TAG": {"IN": ["NNP", "NN"]}, "DEP": "nsubjpass"
            },
             {
                "LEMMA": "be",
                "TAG": "VBZ", "TEXT": "is"
             },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"},
            {"TAG": "VBN"}

            ],
         [
            {

                "TAG": {"IN": ["NNP", "NN"]}, "DEP": "nsubjpass"
            },
             {
                "LEMMA": "be",
                "TAG": "VBZ", "TEXT": "is"},
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"},
            {"TAG": "VBN"}
            ],
         [
            {

                "TAG": {"IN": ["NNP", "NN"]}, "DEP": "nsubjpass"
            },
             {
                "LEMMA": "be",
                "TAG": "VBD", "TEXT": "was"
             },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"},
            {"TAG": "VBN"}
            ],
        [
           {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj"]},
                "OP": "*"
            },
            {

                "TAG": {"IN": ["NNP", "NN"]}
            },
             {
                "LEMMA": "be",
                "TAG": "VBD", "TEXT": "was"
             },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"},
           {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "advmod"]},
                "OP": "*"
            },

            {"TAG": "VBN"}

            ],
        [
           {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj"]},
                "OP": "*"
            },
            {

                "TAG": {"IN": ["NNP", "NN"]}
            },
             {
                "LEMMA": "be",
                "TAG": "VBZ", "TEXT": "is"
             },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"},
           {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "advmod"]},
                "OP": "*"
            },
            {"TAG": "VBN"}

            ],
         [
            {

                "TAG": {"IN": ["NNP", "NN"]},
                "DEP": {"IN": ["nsubj", "nsubjpass"]}
            },
             {
                "LEMMA": "be",
                "TAG": "VBD", "TEXT": "was"
             },
            {"TAG": "RB", "OP": "?"},
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"},
            {"TAG": "VBN"}

            ],
         [
            {

                "TAG": {"IN": ["NNP", "NN"]},
                "DEP": {"IN": ["nsubj", "nsubjpass"]}
            },
             {
                "LEMMA": "be",
                "TAG": "VBZ", "TEXT": "is"
             },
            {"TAG": "RB", "OP": "?"},
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"},
            {"TAG": "VBN"}

            ],
       # it's + VBN (with optional negation/adverb)
[
    {"LOWER": "it"},
     {"LOWER": "'s"},
      {"TAG": {"IN": ["RB","RBS"]}, "OP": "?"},
 {"DEP": "neg", "TAG": "RB", "OP": "?"},
  {"TAG": "VBN"}
],

# it's + VBN + acomp (e.g. "it's considered rude")
[
    {"LOWER": "it"},
     {"LOWER": "'s"},
      {"TAG": "VBN", "LEMMA": "consider"},
       {"DEP": "oprd", "TAG": "JJ"}
    ],
        [
            {"TAG": {"IN": ["NNP", "NN"]}, "DEP": "nsubjpass"},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "aux"]},
                "OP": "*"
            },
             {
                "LEMMA": "be",
                "TAG": "VBD", "TEXT": "was"
             },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"},
           {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "acomp"]},
                "OP": "*"
            },
            {"TAG": "VBN"},
           {
                "DEP": "agent",
                "LOWER": "by"
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "pobj"
            }

            ],
        [
            {"TAG": {"IN": ["NNP", "NN"]}, "DEP": "nsubjpass"},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "aux"]},
                "OP": "*"
            },
             {
                "LEMMA": "be",
                "TAG": "VBZ", "TEXT": "is"
             },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"},
           {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "acomp"]},
                "OP": "*"
            },
            {"TAG": "VBN"},
           {
                "DEP": "agent",
                "LOWER": "by"
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "pobj"
            }

            ],
        [
            {

                "TAG": "CD", "LEMMA": "one"
            },
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct"]},
                "OP": "*"
            },
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBZ"]}
            },
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": {"IN": ["RB", "RBR"]}, "OP": "?"},
            {"TAG": "VBN"}
        ],
        [
            {

                "TAG": {"IN": ["NNP", "NN"]}, "DEP": "nsubjpass"
            },
            {
                "LEMMA": "be",
                "TAG": "VBD", "TEXT": "was"
            },
            {
                "TAG": {"IN": ["RB", "RBS"]},
                "OP": "?"
            },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
            },
            {"TAG": "VBN"}
        ],
         [
            {

                "TAG": {"IN": ["NNP", "NN"]}, "DEP": "nsubjpass"
            },
            {
                "LEMMA": "be",
                "TAG": "VBZ", "TEXT": "is"
            },
            {
                "TAG": {"IN": ["RB", "RBS"]},
                "OP": "?"
            },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
            },
            {"TAG": "VBN"}
        ],
        [
            {

                "TAG": {"IN": ["NNP", "NN"]}, "DEP": "nsubjpass"
            },
            {
                "LEMMA": "be",
                "TAG": "VBZ", "TEXT": "is"
            },
            {
                "TAG": {"IN": ["RB", "RBS"]},
                "OP": "?"
            },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
            },
            {"TAG": "VBN"}
        ],
        [
            {

                "TAG": {"IN": ["NNP", "NN"]}, "DEP": "nsubjpass"
            },
            {
                "LEMMA": "be",
                "TAG": "VBZ", "TEXT": "is"
            },
            {
                "TAG": {"IN": ["RB", "RBS"]},
                "OP": "?"
            },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
            },
            {"TAG": "VBN"}
        ],

[
        {"DEP": "csubjpass"},
        {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "prep", "cc", "conj", "mark", "advcl"]},
                "OP": "*"
        },
        {
            "LEMMA": "be",
            "TAG": "VBD", "TEXT": "was"
        },
        {
          "DEP": "neg",
          "TAG": "RB",
          "OP": "?"
        },
        {
            "TAG": {"IN": ["RB", "RBS"]},
            "OP": "?"
        },
      {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "prep", "cc", "conj", "mark", "advcl"]},
                "OP": "*"
        },
        {"TAG": "VBN"}
        ],
          [
        {
            "DEP": "csubjpass"},
        {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "prep", "cc", "conj", "mark", "advcl"]},
                "OP": "*"
        },
        {
            "LEMMA": "be",
            "TAG": "VBZ", "TEXT": "is"
        },
        {
          "DEP": "neg",
          "TAG": "RB",
          "OP": "?"
        },
        {
            "TAG": {"IN": ["RB", "RBS"]},
            "OP": "?"
        },
      {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "prep", "cc", "conj", "mark", "advcl"]},
                "OP": "*"
        },
        {"TAG": "VBN"}
        ],

        [

    {
        "TAG": {"IN": ["NN", "NNP"]}, "DEP": "nsubjpass"
    },
    {
        "DEP": {"IN": ["prep", "pobj", "det", "amod", "compound"]},
        "OP": "*"
    },
    {
        "LEMMA": "be",
        "TAG": "VBD", "TEXT": "was"
    },
    {
        "TAG": {"IN": ["RB", "RBS"]},
        "OP": "?"
    },
    {
        "DEP": "neg",
        "TAG": "RB",
        "OP": "?"
    },
    {"TAG": "VBN"}
],
         [

    {
        "TAG": {"IN": ["NN", "NNP"]}, "DEP": "nsubjpass"
    },
    {
        "DEP": {"IN": ["prep", "pobj", "det", "amod", "compound"]},
        "OP": "*"
    },
    {
        "LEMMA": "be",
        "TAG": "VBZ", "TEXT": "is"
    },
    {
        "TAG": {"IN": ["RB", "RBS"]},
        "OP": "?"
    },
    {
        "DEP": "neg",
        "TAG": "RB",
        "OP": "?"
    },
    {"TAG": "VBN"}
],
    [
            {

                "TAG": {"IN": ["NNP", "NN", "PRP"]}, "DEP": "nsubjpass"
            },
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "oprd"]},
                "OP": "*"
            },
            {
                "LEMMA": "be",
                "TAG": "VBD", "TEXT": "was"
            },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
            },
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "oprd"]},
                "OP": "*"
            },
            {"TAG": "VBN"}
        ],
        [
            {

                "TAG": {"IN": ["NNP", "NN", "PRP"]}, "DEP": "nsubjpass"
            },
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "oprd"]},
                "OP": "*"
            },
            {
                "LEMMA": "be",
                "TAG": "VBZ", "TEXT": "is"
            },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
            },
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "oprd"]},
                "OP": "*"
            },
            {"TAG": "VBN"}
        ],
        [
            {

                "TAG": {"IN": ["NNP", "NN", "PRP"]},
                "DEP": "nsubjpass"
            },
            {
                "LEMMA": "be",
                "TAG": "VBD",
                "TEXT": "was"
            },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
            },
            {"TAG": "VBN"},
            {
                "POS": "ADV",
                "OP": "?"
            }
            ],

[
            {

                "TAG": {"IN": ["NNP", "NN", "PRP"]}, "DEP": "nsubjpass"
            },
            {
                "LEMMA": "be",
                "TAG": "VBZ", "TEXT": "is"
            },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
            },
            {"TAG": "VBN"},
          {
                "POS": "ADV", "OP": "?"
            }
        ],

[
            {

                "TAG": {"IN": ["NNP", "NN"]},
                "DEP": {"IN": ["nsubj", "nsubjpass"]}
            },
           {
                "DEP": "ROOT",
                "TAG": "VBD", "TEXT": "was"
            },
           {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "npadvmod"]},
                "OP": "*"
            },
            {"TAG": "VBN",
            "DEP": "acomp"}
        ],


[
            {

                "TAG": {"IN": ["NNP", "NN"]},
                "DEP": {"IN": ["nsubj", "nsubjpass"]}
            },
           {
                "DEP": "ROOT",
                "TAG": "VBZ", "TEXT": "is"
            },
           {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "npadvmod"]},
                "OP": "*"
            },
            {"TAG": "VBN",
            "DEP": "acomp"}
        ],
[
            {

                "TAG": {"IN": ["NNP", "NN"]},
                "DEP": {"IN": ["nsubj", "nsubjpass"]},
            },
          {
                "DEP": "auxpass",
                "TAG": {"IN": ["VBD", "VBZ"]}
           },
            {
                "TAG": "VBN",
                "DEP": {"IN": ["ROOT", "advcl", "ccomp"]}
            }
        ],
        [
            {
                "DEP": {"IN": ["nsubjpass", "nsubj"]},
                "TAG": {"IN": ["NN", "NNP"]}},
             {
                "DEP": "auxpass",
                "TAG": {"IN": ["VBD", "VBZ"]}
           },
            {
                "TAG": "VBN"
            },
             {
                "DEP": "agent",
                "LOWER": "by"
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "pobj"]},
                "OP": "*"
            }

        ],
#2 pararrel - singular
      [
            {

                "TAG": {"IN": ["NNP", "NN"]},
                "DEP": {"IN": ["nsubj", "nsubjpass"]},
            },
           {
                "DEP": "ROOT",
                "TAG": {"IN": ["VBD", "VBZ"]}
           },
           {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "conj", "prep"]},
                "OP": "*"
            },
           {"DEP": "cc"},
           {
                "DEP": "auxpass",
                "TAG": {"IN": ["VBD", "VBZ"]}
           },
            {"TAG": "VBN",
            "DEP": {"IN": ["conj", "ROOT", "advcl"]}}
        ],
        [
           {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj"]},
                "OP": "*"
            },
            {

                "TAG": {"IN": ["NNP", "NN"]},
                "DEP": {"IN": ["nsubj", "nsubjpass"]},
            },
           {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj"]},
                "OP": "*"
            },
           {
               "DEP": {"IN": ["aux", "acl", "prep", "pobj", "auxpass"]},
               "OP": "*"
               },
           {
               "TAG": "VBN",
               "DEP": "ROOT"},
           {
                "DEP": "agent",
                "LOWER": "by"
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "pobj"]},
                "OP": "*"
            }
        ],
        [
            {"DEP": {"IN": ["nsubj", "nsubjpass"]}, "TAG": {"IN": ["NN", "NNP"]}},
             {
                "DEP": {"IN": ["acl", "prep", "amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "pobj"]},
                "OP": "*"
            },
             {
                "DEP": "auxpass",
                "TAG": {"IN": ["VBD", "VBZ"]}
           },
            {"TAG": "VBN"},
            {
                "DEP": "agent",
                "LOWER": "by"
            },
            {
                "DEP": {"IN": ["aux", "acl", "prep", "amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "pobj"]},
                "OP": "*"
            }
        ],
         [
            {
                "DEP": {"IN": ["nsubjpass", "nsubj"]},
                "TAG": {"IN": ["NN", "NNP", "JJ"]}},
             {
                "DEP": "auxpass",
                "TAG": {"IN": ["VBD", "VBZ"]}
           },
             {
                "TAG": "RB",
                "OP": "?"
            },
            {
                "TAG": "VBN"
            },
             {
                "DEP": "agent",
                "LOWER": "by"
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "pobj"]},
                "OP": "*"
            }

        ],
        #3 VBN cc VBN (conjoined passives: "are steamed or boiled")
[{"TAG": {"IN": ["NNS","NNP","NN","PRP"]}},
 {"LEMMA": "be", "TAG": {"IN": ["VBP","VBD","VBZ"]}},
 {"TAG": "VBN"},
 {"TAG": "CC"},                     # or/and
 {"TAG": "VBN", "DEP": "conj"}],

        #4specific verb - singular

[{"TAG": "PRP", "LOWER": {"IN": ["he", "she", "it"]}},
 {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ"]}},
 {"TAG": "VBN", "LEMMA": "put"},
 {"TAG": "IN", "LEMMA": "in"},
 {"DEP": {"IN": ["dobj", "pobj"]}}
 ],

        #5 be + VBN (Plural)(Pronouns, name, and sing noun), present and past tense
        # PRP plural + be + VBN/JJ (catches acomp, ROOT, JJ variants)
[{"TAG": "PRP", "LOWER": {"IN": ["they", "you", "we", "it"]}},
 {"LEMMA": "be", "TAG": {"IN": ["VBP", "VBD", "VBZ"]}},
 {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
 {"DEP": "neg", "TAG": "RB", "OP": "?"},
 {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
 {"TAG": "VBN"}],
        [
            {
                "TAG": "PRP",
                "LOWER": {"IN": ["they", "we", "you"]}
            },
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBP"]}
            },
            {
                "TAG": {"IN": ["RB", "RBS"]},
                "OP": "?"
            },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
            },
            {"TAG": "VBN"},
             {
                "POS": "ADV", "OP": "?"
            },
             {
                "DEP": "agent",
                "LOWER": "by"
            },
            {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "pobj",
                "OP": "?"
            }
        ],
        [
            {
                "TAG": "PRP",
                "LOWER": {"IN": ["they", "we", "you"]}
            },
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBP"]}
            },
            {
                "TAG": {"IN": ["RB", "RBS"]},
                "OP": "?"
            },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
            },
            {"TAG": "VBN"}
        ],
                [
            {
                "TAG": "PRP",
                "LOWER": {"IN": ["they", "we", "you"]},
                "DEP": {"IN": ["nsubj", "nsubjpass"]}
            },
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBP"]}
            },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
            },
            {"TAG": "VBN"}
        ],
        [
            {
                "DEP": {"IN": ["nsubjpass", "nsubj"]},
                "TAG": "PRP",
                "LOWER": {"IN": ["you", "they", "we"]}
            },
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBP"]},
                "DEP": "auxpass"
            },
            {
                "TAG": {"IN": ["RB", "RBS"]},
                "OP": "?"
            },
            {
                "TEXT": "n't",
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
            },
            {"TAG": "VBN"},
             {
                "DEP": "agent",
                "LOWER": "by"
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj"]},
                "OP": "*"
            },
           {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "pobj",
                "OP": "?"
            }
        ],
         [
            {
                "DEP": {"IN": ["nsubjpass", "nsubj"]},
                "TAG": "PRP",
                "LOWER": {"IN": ["you", "they", "we"]}
            },
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBP"]},
                "DEP": "auxpass"
            },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
            },
            {"TAG": "VBN"},
             {
                "DEP": "agent",
                "LOWER": "by"
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "pobj"]},
                "OP": "*"
            }
        ],
        [
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj"]},
                "OP": "*"
            },
            {

                "TAG": {"IN": ["NNS", "NNPS"]}
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "prep", "pobj", "pcomp"]},
                "OP": "*"
            },
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBP"]}
            },
            {
                "TAG": {"IN": ["RB", "RBS", "RBR"]},
                "OP": "?"
            },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
            },
            {
                "TAG": {"IN": ["RB", "RBS", "RBR"]},
                "OP": "?"
            },
            {"TAG": "VBN"},
            {
                "DEP": "agent",
                "LOWER": "by"
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "pobj"
            }
        ],
         [
            {

                "TAG": {"IN": ["NNS", "NNPS"]}
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "prep", "pobj", "pcomp"]},
                "OP": "*"
            },
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBP"]}
            },
            {"TAG": "VBN"}
         ],
        [
            {

                "TAG": {"IN": ["NNS", "NNPS"]},
                "DEP": {"IN": ["nsubj", "nsubjpass"]}
            },
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBP"]},
                "DEP": {"IN": ["auxpass", "ROOT"]}
            },
            {
                "TAG": "RB", "OP": "?"
            },
            {"TAG": "VBN"}
         ],
        [
            {
                "TAG": "PRP$",
                "OP": "?"
            },
            {

                "TAG": {"IN": ["NNS", "NNPS"]}
            },
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBP"]}
            },
            {
                "TAG": {"IN": ["RB", "RBS"]},
                "OP": "?"
            },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
            },
            {"TAG": "VBN"},
        ],
        [
            {
                "TAG": "DT",
                "TEXT": {"IN": ["these", "those"]}
            },
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBP"]}
            },
            {
                "TAG": {"IN": ["RB", "RBS"]},
                "OP": "?"
            },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
            },
            {"TAG": "VBN"}
        ],
        [
            {

                "TAG": {"IN": ["NNP", "NN"]}
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "appos"]},
                "OP": "*"
            },
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBZ"]},
                "DEP": "auxpass"
            },
            {
                "TAG": {"IN": ["RB", "RBS"]},
                "OP": "?"
            },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
            },
            {"TAG": "VBN"},
            {
                "DEP": "agent",
                "LOWER": "by"
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "pobj"
            }
        ],
         [
           {"LEMMA": {"IN": ["many", "some", "a lot of", "some of"]}},
            {

                "TAG": {"IN": ["NNP", "NN"]}
            },
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "aux"]},
                "OP": "*"
            },
            {"TAG": {"IN": ["RB", "RBR"]}, "OP": "?"},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "acomp"]},
                "OP": "*"
            },
             {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBP"]}
             },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
          },
           {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "acomp", "pobj"]},
                "OP": "*"
            },
            {"TAG": "VBN"},
           {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "acomp", "pobj"]},
                "OP": "*"
            },
           {
                "DEP": "agent",
                "LOWER": "by"
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "pobj"
            }
            ],
        #pararrel - plural
        [
            {"DEP": {"IN": ["nsubj", "nsubjpass"]}, "TAG": {"IN": ["NN", "NNP"]}},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "prep", "pobj"]},
                "OP": "*"
            },
           {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBP"]}
             },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "conj", "acomp"]},
                "OP": "*"
            },
            {"DEP": "cc"},
            {"TAG": "VBN"}

        ],
        [
            {"DEP": "nsubjpass", "TAG": {"IN": ["NNPS", "NNS"]}},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det", "prep", "pobj", "npadvmod", "punct", "advmod", "conj", "acomp"]},
                "OP": "*"
            },
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBP"]}
            },
            {"TAG": "VBN"}
        ],
 #specific verb - plural

[{"TAG": "PRP", "LOWER": {"IN": ["they", "we", "you"]}},
 {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
 {"TAG": "VBN", "LEMMA": "put"},
 {"TAG": "IN", "LEMMA": "in"},
 {"DEP": {"IN": ["dobj", "pobj"]}}
 ],

        [{"LOWER": "to", "TAG": "TO"},
 {"LEMMA": "be", "POS": "AUX"},
 {"TAG": {"IN": ["NN","VB","VBN"]}},   # catches "love" mistagged
 {"TAG": "CC"},
 {"LEMMA": "be", "TAG": "VB"},
 {"TAG": "VBN"}],


        #plural noun, perfect tense
        [
            {
                "DEP": "nsubjpass",
                "TAG": "PRP",
                "LEMMA": {"IN": ["they", "we", "you"]}
            },
            {
                "LEMMA": "have",
                "TAG": {"IN": ["VBD", "VBP"]},
               "DEP": "aux"
            },
             {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {
                "LEMMA": "be",
                "TAG": "VBN",
                "DEP": "auxpass"
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"TAG": "VBN"},
            {"LEMMA": "by", "DEP": "agent"},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "oprd"]}, #object modifier
                "OP": "*"
            }
        ],
        [
            {
                "DEP": "nsubjpass",
                "TAG": "PRP",
                "TEXT": {"IN": ["he", "is", "it"]}
            },
            {
                "LEMMA": "have",
                "TAG": {"IN": ["VBD", "VBZ"]},
                "TEXT": {"IN": ["had", "has"]},
                "DEP": "aux"},

            {
                "LEMMA": "be",
                "TAG": "VBN",
                "DEP": "auxpass"
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"},
            {"LEMMA": "by", "DEP": "agent"},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "oprd"]}, #object modifier
                "OP": "*"
            }
        ],
        [
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["NNS", "NNPS"]}
            },
            {
                "LEMMA": "have",
                "TAG": {"IN": ["VBD", "VBP"]},
                "TEXT": {"IN": ["had", "have"]},
                "DEP": "aux"},
            {
                "LEMMA": "be",
                "TAG": "VBN",
                "DEP": "auxpass"
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"},
            {"LEMMA": "by", "DEP": "agent"},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "oprd"]}, #object modifier
                "OP": "*"
            }
        ],
        [
            {

                "TAG": {"IN": ["NNP", "NN"]}
            },
            {
                "LEMMA": "have",
                "TAG": {"IN": ["VBD", "VBZ"]},
                "TEXT": {"IN": ["had", "has"]},
                "DEP": "aux"},
            {
                "LEMMA": "be",
                "TAG": "VBN",
                "DEP": "auxpass"
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"},
            {"LEMMA": "by", "DEP": "agent"},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "oprd"]}, #object modifier
                "OP": "*"
            }
        ],
        [
            {"TAG": {"IN": ["NNS", "NNPS", "PRP"]}},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj"]},
                "OP": "*"
            },
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBP"]}
            },
            {
                "LEMMA": {"IN": ["use", "tempt"]},
                "TAG": "VBN"
            },
            {
                "LEMMA": "to",
                "TAG": "IN"
            },
            {
                "TAG": {"IN": ["VBG", "VB"]},
                "POS": "VERB"
            }
        ],
         [
            {"TAG": {"IN": ["NN", "NNP", "PRP"]}},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj"]},
                 "OP": "*"},
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBZ"]}
            },
            {
                "LEMMA": {"IN": ["use", "tempt"]},
                "TAG": "VBN"
            },
            {
                "LEMMA": "to",
                "TAG": "IN"
            },
            {
                "TAG": {"IN": ["VBG", "VB"]},
                "POS": "VERB"
            }
        ],
        [
            {"TAG": {"IN": ["NN", "NNP", "PRP"]}},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj"]},
                 "OP": "*"},
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ"]}},
            {
                "LEMMA": "use",
                "TAG": "VBN"
            },
            {"LEMMA": "for", "TAG": "IN"},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "oprd"]},
                 "OP": "*"}
        ],
        [
            {
                "DEP": {"IN": ["nsubjpass", "nsubj"]},
                "TAG": {"IN": ["NNP", "NN"]}},
             {
                "DEP": "auxpass",
                "TAG": {"IN": ["VBD", "VBZ"]}
           },
            {
                "TAG": "VBN"
            },
             {
                "DEP": "agent",
                "LOWER": "by"
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "pobj"]},
                "OP": "*"
            }

        ],
         [
            {
                "DEP": {"IN": ["nsubjpass", "nsubj"]},
                "TAG": {"IN": ["NNP","NNPS"]}},
             {
                "DEP": "auxpass",
                "TAG": {"IN": ["VBD", "VBP"]},
           },
             {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "pobj"]},
                "OP": "*"
            },
            {
                "TAG": "VBN"
            }

        ],
        [
            {
                "DEP": {"IN": ["nsubjpass", "nsubj"]},
                "TAG": {"IN": ["NNS", "NNPS", "NNS"]}},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "pobj"]},
                "OP": "*"
            },
             {
                "DEP": "auxpass",
                "TAG": {"IN": ["VBD", "VBP"]},
           },
            {
                "TAG": "VBN",
                "DEP": "ROOT"
            }

        ],
        #5 supposed to + VB
        [
            {"TAG": {"IN": ["NNS", "NNPS"]}},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj"]},
                "OP": "*"
            },
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBP"]}
            },
            {
                "LEMMA": "suppose",
                "TAG": "VBN"
            },
            {
                "LEMMA": "to",
                "TAG": "TO"
            },
            {
                "TAG": "VB",
                "POS": "VERB"
            }
        ],
[
            {"TAG": "PRP",
             "LOWER": {"IN": ["they", "we", "you"]}
             },
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBP"]}
            },
            {
                "LEMMA": "suppose",
                "TAG": "VBN"
            },
            {
                "LEMMA": "to",
                "TAG": "TO"
            },
            {
                "TAG": "VB",
                "POS": "VERB"
            }
        ],
        [
            {"TAG": {"IN": ["NN", "NNP"]},
             "DEP": {"IN": ["nsubj", "nsubjpass"]}},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj"]},
                "OP": "*"
            },
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBZ"]},
                "DEP": "auxpass"
            },
            {
                "LEMMA": "suppose",
                "TAG": "VBN"
            },
            {
                "LEMMA": "to",
                "TAG": "TO"
            },
            {
                "TAG": "VB",
                "POS": "VERB"
            }
        ],
[
            {"TAG": "PRP",
             "LOWER": {"IN": ["he", "she", "it"]}},
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBZ"]}
            },
            {
                "LEMMA": "suppose",
                "TAG": "VBN"
            },
            {
                "LEMMA": "to",
                "TAG": "TO"
            },
            {
                "TAG": "VB",
                "POS": "VERB"
            }
        ],
        [
            {

                "TAG": {"IN": ["NNP", "NN"]},
                "DEP": {"IN": ["nsubj", "nsubjpass"]}
            },
           {
               "TAG": {"IN": ["VBZ", "VBD",]},
               "DEP": "auxpass"
               },
            {"TAG": "VBN", "LEMMA": "suppose"},
          {"DEP": "aux", "LEMMA": "to"},
            {
                "DEP": "xcomp",
                "TAG": "VB"
            }
        ],


        #6 have been VBN
        [
            {"LEMMA": "have", "POS": "AUX"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"LEMMA": "be", "TAG": "VBN"},
            {"TAG": "VBN"},
        ],
        #7 modal + be + VBN
        [
            {"TAG": "MD"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"LEMMA": "be", "TAG": "VB"},
            {"LEMMA": "relate", "TAG": "VBN"},
            {"LOWER": "to"}
        ],

        #modal + be + vbn
        [
            {"TAG": "MD"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"LEMMA": "be", "TAG": "VB"},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"TAG": "VBN", "DEP": "ROOT"}
        ],
        # modal + be + VBN with a specific preposition
        [
            {"TAG": "MD"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
             {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"LEMMA": "be", "TAG": "VB"},
            {"TAG": "VBN"}
        ],
[
            {
                "DEP": {"IN": ["nsubjpass", "nsubj"]},
                "TAG": {"IN": ["NN", "NNP", "JJ"]}},
             {
                "DEP": "auxpass",
                "TAG": {"IN": ["VBD", "VBZ"]}
           },
             {
                "TAG": "RB",
                "OP": "?"
            },
            {
                "TAG": "VBN",
                "LEMMA": {"IN": ["regard", "see", "view", "perceive", "acknowledge", "describe", "define", "classify", "characterize", "identify", "function", "recognize", "know"]},
                "DEP": "ROOT"
            },
             {
                "DEP": "PREP",
                "LOWER": "as"
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "case", "det",  "npadvmod", "punct", "advmod", "cc", "conj"]},
                "OP": "*"
            },
            {"DEP": "pobj"}

        ],
[
            {
                "DEP": {"IN": ["nsubjpass", "nsubj"]},
                "TAG": {"IN": ["NN", "NNP", "JJ"]}},
             {
                "DEP": "auxpass",
                "TAG": {"IN": ["VBD", "VBZ"]}
           },
             {
                "TAG": "RB",
                "OP": "?"
            },
            {
                "TAG": "VBN", "LEMMA": "locate", "DEP": "ROOT"
            },
             {
                "DEP": "PREP",
                "LOWER": "in"
            },
            {
                "DEP": {"IN": ["amod", "nmod", "compound", "poss", "nummod", "case", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "pobj"]},
                "OP": "*"
            }

        ],

        # S + MD


        [

            {"TAG": {"IN": ["PRP", "NNS", "NNPS"]}},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["WDT", "WP"]},
                "OP": "?"
            },
            {"TAG": "MD"},
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
            },
            {
                "TAG": {"IN": ["RB", "RBS"]},
                "OP": "?"
            },
            {
                "LEMMA": "be",
                "TAG": "VB"
           },
            {"TAG": "VBN"}
        ],
        # cannot + be + VBN
        [
            {"TEXT": "cannot"},
            {"LEMMA": "be", "TAG": "VB"},
            {"TAG": "VBN"}
        ],
        #8 to be VBN
        [
            {"LOWER": "to", "TAG": "TO"},
            {"TAG": "RB", "OP": "?"},
            {"LEMMA": "be", "POS": "AUX"},
            {"TAG": "VBN"}
        ],
        #9 plural + who + be + VBN
        [
            {
                "TAG": {"IN": ["NNS", "NNPS"]}
            },
           {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {"TAG": {"IN": ["WP", "WDT"]}},
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBP"]}
            },
            {
                "TAG": {"IN": ["RB", "RBS"]},
                "OP": "?"
            },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
            },
            {"TAG": "VBN"}
        ],
        #8 SINGULAR + WHO + BE + VBN
        [
            {
                "TAG": {"IN": ["NN", "NNP"]}, "DEP": "nsubjpass"
            },
            {"TAG": {"IN": ["WP", "WDT"]}},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det", "advmod", "npadvmod", "punct", "advmod", "cc", "conj", "amod", "attr", "acl", "relcl", "dobj"]},
                "OP": "*"
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ"]}, "DEP": "auxpass"},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det", "advmod", "npadvmod", "punct", "advmod", "cc", "conj", "amod", "attr", "acl"]},
                "OP": "*"
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
        [
            {
                "TAG": "PRP",
                "LOWER": {"IN": ["she", "he", "it"]}
            },
            {"TAG": {"IN": ["WP", "WDT"]}},
             {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det", "advmod", "npadvmod", "punct", "advmod", "cc", "conj", "amod", "attr", "acl", "relcl", "dobj"]},
                "OP": "*"
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ"]}},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det", "advmod", "npadvmod", "punct", "advmod", "cc", "conj", "amod", "attr", "acl"]},
                "OP": "*"
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],

        #12 PLURAL + WHICH + BE + VBN
        [
             {
                "TAG": {"IN": ["NNS", "NNPS"]}
            },
            {"TAG": "WDT"},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det", "advmod", "npadvmod", "punct", "advmod", "cc", "conj", "amod", "attr", "acl", "relcl", "dobj"]},
                "OP": "*"
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
        [
             {
                "TAG": "PRP",
                "LOWER": {"IN": ["they", "we", "you"]}
            },
            {"TAG": "WDT"},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det", "advmod", "npadvmod", "punct", "advmod", "cc", "conj", "amod", "attr", "acl", "relcl", "dobj"]},
                "OP": "*"
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
        [
            {
                "TAG": "NNS"
            },
            {"TAG": "WP"},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det", "advmod", "npadvmod", "punct", "advmod", "cc", "conj", "amod", "attr", "acl", "relcl", "dobj"]},
                "OP": "*"
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det", "advmod", "npadvmod", "punct", "advmod", "cc", "conj", "amod", "attr", "acl", "relcl", "dobj"]},
                "OP": "*"
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
        #13 SING + WHICH + BE + VBN
        [
            {
                "TAG": "PRP",
                "LOWER": {"IN": ["she", "he", "it"]}
            },
            {"TAG": "WDT"},
             {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det", "advmod", "npadvmod", "punct", "advmod", "cc", "conj", "amod", "attr", "acl", "relcl", "dobj"]},
                "OP": "*"
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ"]}},
             {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det", "advmod", "npadvmod", "punct", "advmod", "cc", "conj", "amod", "attr", "acl", "relcl", "dobj"]},
                "OP": "*"
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
        [
            {
                "TAG": {"IN": ["NN", "NNP"]}
            },
            {"TAG": "WDT"},
             {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det", "advmod", "npadvmod", "punct", "advmod", "cc", "conj", "amod", "attr", "acl", "relcl", "dobj"]},
                "OP": "*"
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ"]}},
             {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det", "advmod", "npadvmod", "punct", "advmod", "cc", "conj", "amod", "attr", "acl", "relcl", "dobj"]},
                "OP": "*"
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
        [
            {
                "TAG": "NNS"
            },
            {"TAG": "WDT"},
             {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det", "advmod", "npadvmod", "punct", "advmod", "cc", "conj", "amod", "attr", "acl", "relcl", "dobj"]},
                "OP": "*"
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
             {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det", "advmod", "npadvmod", "punct", "advmod", "cc", "conj", "amod", "attr", "acl", "relcl", "dobj"]},
                "OP": "*"
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
        [
            {"TAG": "WDT"},
             {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det", "advmod", "npadvmod", "punct", "advmod", "cc", "conj", "amod", "attr", "acl", "relcl", "dobj"]},
                "OP": "*"
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
             {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det", "advmod", "npadvmod", "punct", "advmod", "cc", "conj", "amod", "attr", "acl", "relcl", "dobj"]},
                "OP": "*"
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"},
            {"DEP": "agent", "LOWER": "by"},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "pobj"
            }
        ],
        [
            {"TAG": "WDT"},
             {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det", "advmod", "npadvmod", "punct", "advmod", "cc", "conj", "amod", "attr", "acl", "relcl", "dobj"]},
                "OP": "*"
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ"]}},
             {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det", "advmod", "npadvmod", "punct", "advmod", "cc", "conj", "amod", "attr", "acl", "relcl", "dobj"]},
                "OP": "*"
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],

        #14 VERB + BEING + VBN
        [
            {"POS": "VERB"},
            {
                "LEMMA": "be",
                "TAG": "VBG",
                "DEP": "auxpass"
            },
            {
                "TAG": "VBN",
                "DEP": "xcomp"
                }

        ],
        [
            {
                "TAG": {"IN": ["NNS", "NNPS"]}
            },
            {"POS": "AUX", "TAG": "VBP"},
           {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"POS": "VERB", "TAG": "VB"},
            {
                "LEMMA": "be",
                "TAG": "VBG"
            },
            {
                "TAG": "VBN",
                "DEP": "xcomp"
                }
        ],
        [
            {
                "TAG": "PRP",
                "LOWER": {"IN": ["they", "we", "you"]},
                "DEP": {"IN": ["nsubj", "nsubjpass"]}
            },
            {"DEP": "aux", "LEMMA": "do", "TAG": "VBP"},
           {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"POS": "VERB", "TAG": "VB"},
            {
                "TAG": "VBG",
                "DEP": "auxpass"
            },
            {
                "TAG": "VBN",
                "DEP": "xcomp"
                }
        ],

      [

            {
                "TAG": {"IN": ["NN", "NNP"]}
            },
            {"POS": "AUX", "LEMMA": "do", "TAG": "VBZ"},
           {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"POS": "VERB"},
            {
                "LEMMA": "be",
                "TAG": "VBG",
                "DEP": "auxpass"
            },
            {
                "TAG": "VBN",
                "DEP": "xcomp"
                }
        ],

        [
            {
                "TAG": "PRP"
            },
            {"POS": "AUX", "LEMMA": "do", "TAG": "VBZ"},
           {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"POS": "VERB"},
            {
                "LEMMA": "be",
                "TAG": "VBG",
                "DEP": "auxpass"
            },
            {
                "TAG": "VBN",
                "DEP": "xcomp"
                }
        ],
         [
            {
                "TAG": "PRP",
                "LOWER": {"IN": ["they", "we", "you"]}

            },
            {"DEP": "aux", "LEMMA": "do", "TAG": "VBP"},
           {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VB", "DEP": "ROOT"},
            {
                "LEMMA": "be",
                "TAG": "VBG",
                "DEP": "auxpass"
            },
            {
                "TAG": "VBN",
                "DEP": "xcomp"
                }
        ],
 [
            {
                "TAG": {"IN": ["NN", "NNP", "PRP", "NNS", "NNPS"]}
            },
            {"POS": "AUX", "LEMMA": "do", "TAG": "VBD"},
           {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"POS": "VERB"},
            {
                "LEMMA": "be",
                "TAG": "VBG",
                "DEP": "auxpass"
            },
            {
                "TAG": "VBN",
                "DEP": "xcomp"
                }
        ],
        [{"TAG": "DT"}, {"TAG": {"IN": ["NN","NNP"]},"OP":"+"}, {"LEMMA":"be", "DEP": {"IN": ["VBD", "VBZ"]}}, {"TAG":"VBN"}],


        #15 BE + RELATED / TAKEN / EXPOSE / LINK + TO
        [
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {

                "TAG": "NN",
                "DEP": {"IN": ["nsubjpass", "nsubj"]}
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"LEMMA": {"IN": ["relate", "take", "expose", "link"]}, "TAG": "VBN"},
            {"LEMMA": "to", "DEP": "prep"},
           {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "pobj"
            }
        ],
         [
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {

                "TAG": {"IN": ["NNPS", "NNS"]}
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"LEMMA": {"IN": ["relate", "take", "expose", "link"]}, "TAG": "VBN"},
            {"LEMMA": "to", "DEP": "prep"},
           {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "case", "conj", "amod"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "pobj"
            }
        ],
        [
            {

                "TAG": {"IN": ["NNS", "NNPS"]}
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"LEMMA": {"IN": ["relate", "take"]}, "TAG": "VBN"},
            {"LEMMA": "to", "TAG": "IN"},
           {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            }
        ],
 [
            {

                "TAG": "PRP",
                "LOWER": {"IN": ["she", "he", "it"]}
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"LEMMA": {"IN": ["relate", "take"]}, "TAG": "VBN"},
            {"LEMMA": "to", "TAG": "IN"},
           {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod", "case"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "pobj"
            }
        ],
[
            {

                "TAG": "PRP",
                "LOWER": {"IN": ["they", "you", "we"]}
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBP", "VBD"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"LEMMA": {"IN": ["relate", "take"]}, "TAG": "VBN"},
            {"LEMMA": "to", "TAG": "IN"},
           {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod", "case"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "pobj"
            }
        ],

        #16 based + on ...
         [
            {

                "TAG": "NN"
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"LEMMA": "base", "TAG": "VBN"},
            {"LEMMA": "on", "TAG": "IN"},
           {"DEP": "pobj", "OP": "?"}
        ],
        [
           {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {

                "TAG": "NN"
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"LEMMA": "base", "TAG": "VBN"},
            {"LEMMA": "on", "TAG": "IN"},
            {"DEP": "pobj"}
        ],
         #17 most of ...
        [
            {"DEP": "nsubjpass", "LEMMA": "most"},
             {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "prep", "pobj", "acomp", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {
              "TAG": "NNS"
              },
            {"LEMMA": "be", "TAG": {"IN":["VBD", "VBP"]}},
            {
                "TAG": "VBN",
                "POS": "VERB"
            }
        ],

        #18 imperative
        [{"LEMMA": "do", "TAG": "VB"},   # no DEP constraint
 {"DEP": "neg", "TAG": "RB"},
 {"LEMMA": "be", "TAG": "VB"},   # no DEP constraint
 {"TAG": "VBN"}],
        [
 {"LEMMA": "do"},
 {"DEP": "neg"},
 {"LEMMA": "be"},
 {"TAG": "VBN"}
],
  #DO NOT BE VBN
        [
            {"LEMMA": "do", "POS": "AUX", "TAG": "VB", "DEP": "aux"},
             {"DEP": "neg", "TAG": "RB"},
            {"TAG": "VB", "LEMMA": "be", "DEP": "auxpass"},
            {"TAG": "VBN"}
        ],
        #19 THAT IS CALLED
        [
            {
                "TAG": "WDT",
                "LEMMA": "that",
                "DEP": "nsubjpass"
            },

               {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {
                "LEMMA": "call",
                "TAG": "VBN"
            },
            {"TAG": {"IN": ["NN", "NNS"]}}
        ],

        #it's called
        [
            {
                "TAG": "PRP",
                "LOWER": {"IN": ["he", "she", "it"]},
                "DEP": "nsubjpass"
            },

               {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ"]}},
            {
                "LEMMA": "call",
                "TAG": "VBN"
            },
            {
                "TAG": {"IN": ["NN", "NNS", "NNP"]},
                "DEP": "dobj"
                }
        ],
        #they're called
        [
            {
                "TAG": "PRP",
                "LOWER": {"IN": ["you", "they", "we"]},
                "DEP": "nsubjpass"
            },

               {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {
                "LEMMA": "call",
                "TAG": "VBN"
            },
            {
                "TAG": {"IN": ["NN", "NNS", "NNP"]},
                "DEP": "dobj"
                }
        ],

      #20 GETS + VBN
        [
            {

                "TAG": {"IN": ["NNP", "NN"]}
            },
            {
                "LEMMA": "get",
                "TAG": "VBZ",
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
         [
            {

                "TAG": {"IN": ["NNP", "NN"]}
            },
            {
                "LEMMA": "get",
                "TAG": "VBZ"
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
 #17 GET + VBN
       [
            {

                "TAG": "NNS"
            },
           {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "prep", "pobj", "acomp", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {
                "LEMMA": "get",
                "TAG": {"IN":["VBP", "VB"]}
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
         [
            {

                "TAG": "NNS"
            },
            {"TAG": "MD"},
            {
                "LEMMA": "get",
                "TAG": "VB",
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"TAG": "VBN"}
        ],
        [
            {

                "TAG": "PRP",
                "LOWER": {"NOT_IN": ["he", "she", "it"]}
            },
            {
                "LEMMA": "get",
                "TAG": {"IN":["VB", "VBP"]}
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"TAG": "VBN"}
        ],
        [
            {
                "TAG": "PRP",
                "LOWER": {"IN": ["he", "she", "it", "I"]},
            },
            {
                "LEMMA": "be",
                "TAG": "VBD", "TEXT": "was"},
               {
                "LEMMA": "get",
                "TAG": "VBG"},
            {"TAG": "VBN"}
        ],
        [
            {
                "TAG": "PRP",
                "LOWER": {"IN": ["he", "she", "it"]},
            },
            {
                "LEMMA": "be",
                "TAG":   "VBZ"},
               {
                "LEMMA": "get",
                "TAG": "VBG"},
            {"TAG": "VBN"}
        ],
[
            {
                "TAG": "PRP",
                "LOWER": {"IN": ["I", "you", "they", "we"]},
            },
            {
                "LEMMA": "be",
                "TAG":   "VBP"},
               {
                "LEMMA": "get",
                "TAG": "VBG"},
            {"TAG": "VBN"}
        ],
        [
            {
                "TAG": "PRP",
                "LOWER": {"IN": ["you", "they", "we"]},
            },
            {
                "LEMMA": "be",
                "TAG":   "VBD", "TEXT": "were"},
               {
                "LEMMA": "get",
                "TAG": "VBG"},
            {"TAG": "VBN"}
        ],
        [
            {

                "TAG": "PRP"
            },
            {"TAG": "MD"},
            {
                "LEMMA": "get",
                "TAG": {"IN":["VB", "VBP"]}
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"TAG": "VBN"}
        ],
        [
            {
    "TAG": "PRP",
    "LOWER": {"NOT_IN": ["he", "she", "it"]}
},
            {
                "LEMMA": "get",
                "TAG": {"IN":["VB", "VBP"]}
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {
                "TAG": "VBN",
                "LEMMA": "use"
            },
            {
                "TAG": "TO",
                "LEMMA": "to"
            }
        ],
        [
         {
    "TAG": "PRP",
    "LOWER": {"NOT_IN": ["I", "you", "they", "we"]}
},
            {
                "LEMMA": "get",
                "TAG": "VBZ"
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {
                "TAG": "VBN",
                "LEMMA": "use"
            },
            {
                "TAG": "TO",
                "LEMMA": "to"
            }
        ],
        [
            {"LEMMA": "to", "TAG": "TO"},
            {"LEMMA": "get",
             "TAG": "VB"},
            {"TAG": "VBN"}
        ],

         #18 GOT + VBN
        [
            {

                "TAG": {"IN": ["NNP", "NN"]}
            },
            {
                "LEMMA": "get",
                "TAG": "VBD",
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
 [
            {

                "TAG": {"IN": ["NNS", "NNPS"]}
            },
            {
                "LEMMA": "get",
                "TAG": "VBD",
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],

        #MD + GET
        [
            {"TAG": "MD"},
            {
                "LEMMA": "get",
                "TAG": "VBP"
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],

        #GETTING + VBN
        [
            {"DEP": "nsubjpass", "TAG": "PRP", "LEMMA": {"IN": ["he", "she", "it", "I"]}},
            {"TAG": "VBD", "LEMMA": "be", "TEXT": "was"},
            {"LEMMA": "getting", "TAG": "VBG"},
            {"TAG": "VBN"}
                                                                ],
            [
            {"DEP": "nsubjpass", "TAG": "PRP", "LEMMA": {"IN": ["he", "she", "it"]}},
            {"TAG": "VBZ", "LEMMA": "be", "TEXT": "is"},
            {"LEMMA": "get", "TAG": "VBG"},
            {"TAG": "VBN"}
                                                                ],

[
            {"DEP": "nsubjpass", "TAG": "PRP", "LEMMA": {"IN": ["you", "they", "we"]}},
            {"TAG": "VBP", "LEMMA": "be", "TEXT": "are"},
            {"LEMMA": "get", "TAG": "VBG"},
            {"TAG": "VBN"}
                                                                ],
[
            {"DEP": "nsubjpass", "TAG": "PRP", "LEMMA": {"IN": ["you", "they", "we"]}},
            {"TAG": "VBD", "LEMMA": "be", "TEXT": "were"},
            {"LEMMA": "get", "TAG": "VBG"},
            {"TAG": "VBN"}
                                                                ],
        [
            {"DEP": "nsubjpass", "TAG": "PRP", "LEMMA": "I"},
            {"TAG": "VBP", "LEMMA": "be", "TEXT": "am"},
            {"LEMMA": "get", "TAG": "VBG"},
            {"TAG": "VBN"}
                                                                ],
        [
            {"DEP": "nsubjpass", "TAG": {"IN": ["NNS", "NNPS"]}},
            {"TAG": "VBD", "LEMMA": "be", "TEXT": "were"},
            {"LEMMA": "get", "TAG": "VBG"},
            {"TAG": "VBN"}
                                                                ],
        [
            {"DEP": "nsubjpass", "TAG": {"IN": ["NNS", "NNPS"]}},
            {"TAG": "VBP", "LEMMA": "be", "TEXT": "are"},
            {"LEMMA": "get", "TAG": "VBG"},
            {"TAG": "VBN"}
                                                                ],


        #21 gerund

        [
          {"DEP": "csubjpass", "TAG": "VBG"},
          {
                "DEP": {"IN": ["dobj", "pobj"]},
                "OP": "*"
          },
          {
              "LEMMA": "be",
              "TAG": {"IN": ["VBD", "VBZ"]},
              "DEP": "auxpass"
          },
          {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
          {"DEP": "neg", "TAG": "RB", "OP": "?"},
          {"TAG": "VBN"}

        ],


        #22 expletive "there is" and "there are"
        [
            {"DEP": "expl"},
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ"]}},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {"TAG": {"IN": ["NNS", "NNPS"]}, "DEP": "attr"},
            {"DEP": "nsubjpass"},
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP",]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
        [
            {"DEP": "expl"},
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {"TAG": {"IN": ["NNS", "NNPS"]}, "DEP": "attr"},
            {"DEP": "nsubjpass"},
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],

        #23 PASSIVE PROGRESSIVE
        [
            {
               "DEP": {"IN": ["nsubj", "nsubjpass", "csubjpass"]},
               "TAG": {"IN": ["NN", "NNP"]}
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBG", "TEXT": "being"},
            {"TAG": "VBN"}
        ],
                [
            {
               "DEP": {"IN": ["nsubj", "nsubjpass", "csubjpass"]},
               "TAG": "PRP",
               "LOWER": {"IN": ["he", "she", "it"]}
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBG", "TEXT": "being"},
            {"TAG": "VBN"}
        ],
        [
            {
               "DEP": {"IN": ["nsubj", "nsubjpass", "csubjpass"]},
               "TAG": {"IN": ["NNS", "NNPS"]}
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
           {"TAG": "VBG", "TEXT": "being"},
            {"TAG": "VBN"}
        ],
        [
            {
               "DEP": {"IN": ["nsubj", "nsubjpass", "csubjpass"]},
               "TAG": "PRP",
              "LOWER": {"IN": ["they", "we", "you"]}
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
           {"TAG": "VBG", "TEXT": "being"},
            {"TAG": "VBN"}
        ],
[
            {
                "TAG": {"IN": ["NN", "NNP"]}
            },
            {"TAG": {"IN": ["VBZ", "VBD"]}},
        {"LEMMA": "be", "TAG": "VBG"},
        {"TAG": "VBN"}
        ],
        [
             {
                "TAG": {"IN": ["NN", "NNP"]},
                "DEP": {"IN": ["nsubjpass", "nsubj"]}
            },
            {"TAG": {"IN": ["VBZ", "VBD"]}},
        {"LEMMA": "be", "TAG": "VBG", "DEP": "auxpass"},
        {"TAG": "VBN", "DEP": "ROOT"}
        ],
        [
            {
                "TAG": {"IN": ["NNS", "NNPS"]}
            },
            {"TAG": {"IN": ["VBP", "VBD"]}},
        {"LEMMA": "be", "TAG": "VBG"},
        {"TAG": "VBN"}
        ],
        [
            {"DEP": {"IN": ["nsubjpass", "nsubj"]}, "DEP": {"IN": ["NN", "NNP"]}},
            {
                "TAG": {"IN": ["VBZ", "VBD"]},
                "DEP": "aux"},
            {
                "DEP": "auxpass",
                "TAG": "VBG",
                "LEMMA": "be"
            },
            {"TAG": "VBN"}
        ],

        #24 relative clause modifier
        [
            {"DEP": "nsubj"},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "relcl"},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct"]}, #object modifier
                "OP": "*"
            },
           {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
        [
            {"DEP": "relcl"},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct"]}, #object modifier
                "OP": "*"
            },
           {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
        [

             {

                "TAG": {"IN": ["NN", "NNP"]}
            },
            {"DEP": "nsubj"},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "relcl"},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct"]}, #object modifier
                "OP": "*"
            },
           {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
        [

            {"DEP": "relcl"},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct"]}, #object modifier
                "OP": "*"
            },
           {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],

        #25 object predicate complement
        [
            {

                "TAG": {"IN": ["NNP", "NN", "JJ"]}
            },
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBZ"]},
                "DEP": "auxpass"
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {
                "TAG": "VBN",
                "LEMMA": {"IN": [
    "call", "name", "label", "nickname", "title", "dub", "christen", "designate",
    "make", "render", "leave", "keep", "drive", "turn", "get", "set",
    "consider", "regard", "deem", "judge", "declare", "find", "believe", "think",
    "elect", "appoint", "select", "choose", "vote", "promote",
    "prove", "show", "demonstrate",
    "paint", "portray", "describe", "depict"
]}
                },
           {"DEP": {"IN": ["oprd", "dobj", "pobj"]}}
        ],
        [
            {

                "TAG": "PRP",
                 "LOWER": {"IN": ["he", "she", "it"]}
            },

            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBZ"]},
                "DEP": "auxpass"
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {
                "TAG": "VBN",
                "LEMMA": {"IN": [
    "call", "name", "label", "nickname", "title", "dub", "christen", "designate",
    "make", "render", "leave", "keep", "drive", "turn", "get", "set",
    "consider", "regard", "deem", "judge", "declare", "find", "believe", "think",
    "elect", "appoint", "select", "choose", "vote", "promote",
    "prove", "show", "demonstrate",
    "paint", "portray", "describe", "depict"
]}
                },
           {"DEP": {"IN": ["oprd", "dobj", "pobj"]}}
        ],
         [
            {

                "TAG": "PRP",
                "LOWER": {"IN": ["he", "she", "it"]}
            },

            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBZ"]},
                "DEP": "auxpass"
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {
                "TAG": "VBN",
                "LEMMA": {"IN": [
    "call", "name", "label", "nickname", "title", "dub", "christen", "designate",
    "make", "render", "leave", "keep", "drive", "turn", "get", "set",
    "consider", "regard", "deem", "judge", "declare", "find", "believe", "think",
    "elect", "appoint", "select", "choose", "vote", "promote",
    "prove", "show", "demonstrate",
    "paint", "portray", "describe", "depict"
]}
                },
           {"DEP": "oprd"}
        ],

        [
            {

                "TAG": {"IN": ["NNS", "NNPS"]}
            },

            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBP"]},
                "DEP": "auxpass"
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {
                "TAG": "VBN",
                "LEMMA": {"IN": [
    "call", "name", "label", "nickname", "title", "dub", "christen", "designate",
    "make", "render", "leave", "keep", "drive", "turn", "get", "set",
    "consider", "regard", "deem", "judge", "declare", "find", "believe", "think",
    "elect", "appoint", "select", "choose", "vote", "promote",
    "prove", "show", "demonstrate",
    "paint", "portray", "describe", "depict"
]}
                },
           {"DEP": "oprd"}
        ],
 [
            {

                "TAG": "PRP",
                "LOWER": {"IN": ["you", "they", "we"]}
            },

            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBP"]},
                "DEP": "auxpass"
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {
                "TAG": "VBN",
                "LEMMA": {"IN": [
    "call", "name", "label", "nickname", "title", "dub", "christen", "designate",
    "make", "render", "leave", "keep", "drive", "turn", "get", "set",
    "consider", "regard", "deem", "judge", "declare", "find", "believe", "think",
    "elect", "appoint", "select", "choose", "vote", "promote",
    "prove", "show", "demonstrate",
    "paint", "portray", "describe", "depict"
]}
                },
           {"DEP": "oprd"}
        ],

#causative passive
        [
            {"DEP": "JJ"},
            {"DEP": "cc"},
            {"DEP": "JJ"},
            {"TAG": {"IN": ["VBP", "VBD", "VB"]}, "LEMMA": {"IN": ["make", "have", "get", "need", "want", "keep", "leave"]}},
             {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct"]}, #object modifier
                "OP": "*"
            },
            {"DEP": "nsubj"},
            {"TAG": "VBN", "DEP": "ccomp"}
        ],
        [
            {"DEP": "nsubj", "TAG": "PRP", "LEMMA": {"IN": ["I", "you", "they", "we"]}},
            {"TAG": {"IN": ["VBP", "VBD", "VB"]}, "LEMMA": {"IN": ["make", "have", "get", "need", "want", "keep", "leave"]}},
             {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct"]}, #object modifier
                "OP": "*"
            },
            {"DEP": "nsubj"},
            {"TAG": "VBN", "DEP": "ccomp"}
        ],
        [
            {"DEP": "nsubj", "TAG": "PRP", "LEMMA": {"IN": ["she", "he", "it"]}},
            {"TAG": {"IN": ["VBZ", "VBD", "VB"]}, "LEMMA": {"IN": ["make", "have", "get", "need", "want", "keep", "leave"]}},
             {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct"]}, #object modifier
                "OP": "*"
            },
            {"DEP": "nsubj"},
            {"TAG": "VBN", "DEP": "ccomp"}
        ],
        [
            {"DEP": "nsubj", "TAG": {"IN": ["NN", "NNP"]}},
            {"TAG": {"IN": ["VBZ", "VBD", "VB"]}, "LEMMA": {"IN": ["make", "have", "get", "need", "want", "keep", "leave"]}},
             {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct"]}, #object modifier
                "OP": "*"
            },
            {"DEP": "nsubj"},
            {"TAG": "VBN", "DEP": "ccomp"}
        ],
        [
            {"DEP": "nsubj", "TAG": {"IN": ["NNS", "NNPS"]}},
            {"TAG": {"IN": ["VBP", "VBD", "VB"]}, "LEMMA": {"IN": ["make", "have", "get", "need", "want", "keep", "leave"]}},
             {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct"]}, #object modifier
                "OP": "*"
            },
            {"DEP": "nsubj"},
            {"TAG": "VBN", "DEP": "ccomp"}
        ],

      #compound subject
       # Compound subject — unlimited nouns
[
    {"TAG": {"IN": ["NNS", "NNPS", "NN", "NNP"]}},   # first noun
    {"TAG": ",", "OP": "?"},                           # optional comma
    {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "relcl", "acl", "appos", "cc", "conj", "punct"]}, #object modifier
                "OP": "*"
            },
    {"TAG": {"IN": ["NNS", "NNPS", "NN", "NNP"]}, "OP": "*"},  # more nouns
    {"TAG": "CC"},                                     # final "and/or" (required)
      {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "relcl", "acl", "appos", "cc", "conj", "punct"]}, #object modifier
                "OP": "*"
            },
    {"TAG": {"IN": ["NNS", "NNPS", "NN", "NNP"]}},   # last noun (required)
    {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
    {"TAG": "VBN"}
]

    ]

    matcher.add("CorrectPassiveVoice", patterns)
    matches = matcher(doc, as_spans=True)
    matches = spacy.util.filter_spans(matches)

    # Removed the recursive call: if correct(new_text) > 0:

    # ── Filter 1: Remove "do not be VBN" with no subject (= correct imperative) ──
    has_subject = any(t.dep_ in ["nsubj", "nsubjpass"] for t in doc)
    has_do_not_be = any(t.lemma_ == "do" and t.tag_ == "VB" for t in doc)

    if has_do_not_be and not has_subject:
        return 0 # Return 0 correct if it's this specific imperative error

    # ── Filter 2: Remove matches that have a dobj (= not a true passive) ──
    filtered = []
    for span in matches:
        span_token_ids = {token.i for token in span}
        has_dobj = False
        for token in doc:
            if token.dep_ == "dobj" and token.head.i in span_token_ids:
                has_dobj = True
                break
        if not has_dobj:
            filtered.append(span)

    # The verbose print statement about 'Error passive voice usages detected' was removed as it's not relevant for the correct() function.

    return len(filtered) # Return the count of filtered correct passive usages

def error(new_text, verbose=False):
    doc = nlp(new_text)
    matcher = Matcher(nlp.vocab)

    if verbose:
        for token in doc:
            print(token.text, token.lemma_, token.pos_, token.tag_, token.dep_, token.head.text)

    patterns = [

          #1 Singular Noun + AUX have
          [
              {
                  "TAG": {"IN": ["NN", "NNP"]}
            },
            {"LEMMA": "have", "POS": "AUX", "TAG": "VBP"},
            {"LEMMA": "be", "TAG": "VBN"},
            {"POS": "ADV", "OP": "?"},
            {"TAG": "VBN"}
        ],
        #2 noun non human in active voice
        [
             {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {"DEP": "nsubj", "TEXT": {"IN": ["rules", "concepts", "topic", "topics", "thing", "things", "issue", "issues",
    "matter", "matters", "subject", "subjects", "question", "questions",
    "problem", "problems", "fact", "facts", "idea", "ideas",
    "concept", "concepts", "point", "points", "aspect", "aspects",
    "result", "results", "effect", "effects", "reason", "reasons",
    "case", "cases", "situation", "situations", "example", "examples", "things"]}},
             {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct"]}, #object modifier
                "OP": "*"
            },
            {"TAG": {"IN": ["VBD", "VBD"]}},
            {"TAG": "IN"},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "oprd"]}, #object modifier
                "OP": "*"
            }

        ],
        [
            {"DEP": "nsubj", "TEXT": {"IN": ["topic", "rule", "concept", "thing", "topics", "thing", "things", "issue", "issues",
    "matter", "matters", "subject", "subjects", "question", "questions",
    "problem", "problems", "fact", "facts", "idea", "ideas",
    "concept", "concepts", "point", "points", "aspect", "aspects",
    "result", "results", "effect", "effects", "reason", "reasons",
    "case", "cases", "situation", "situations", "example", "examples"]}},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct"]}, #object modifier
                "OP": "*"
            },
            {"TAG": {"IN": ["VBD", "VBZ"]}},
            {"TAG": "IN"},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct"]}, #object modifier
                "OP": "*"
            }

        ],
        [

            {"DEP": "nsubj", "TEXT": {"IN": ["topic", "topics", "thing", "things", "issue", "issues",
    "matter", "matters", "subject", "subjects", "question", "questions",
    "problem", "problems", "fact", "facts", "idea", "ideas",
    "concept", "concepts", "point", "points", "aspect", "aspects",
    "result", "results", "effect", "effects", "reason", "reasons",
    "case", "cases", "situation", "situations", "example", "examples", "things"]}},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct"]}, #object modifier
                "OP": "*"
            },
            {"TAG": "MD"},
            {"TAG": "VB"}

        ],


        #4 Double be
        [
        {
            "TAG": {"IN": ["NNS", "NN"]}
        },
        {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ", "VBP", "VBN"]}},
        {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ", "VBP", "VBN"]}},
        {"TAG": "RB", "OP": "*"},
        {"TAG": "VBN"}
        ],
          #DOUBLE VERB
          [
        {
            "TAG": {"IN": ["NNS", "NN", "NNP", "NNPS"]}, "DEP": {"IN": ["nsubj", "nsubjpass"]}
        },
        {"POS": "VERB"},
        {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ", "VBP", "VBN"]}},
        {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ", "VBP", "VBN"]}},
        {"TAG": "RB", "OP": "*"},
        {"TAG": "VBN"}
        ],
           [
        {
           "DEP": {"IN": ["nsubj", "nsubjpass"]}
        },
        {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ", "VBP", "VBN"]}},
        {"TAG": "VBG"},
        {"TAG": "RB", "OP": "?"},
        {"TAG": "VBN"}
        ],
          [
        {
            "TAG": {"IN": ["NNS", "NN", "NNP", "NNPS"]}, "DEP": {"IN": ["nsubj", "nsubjpass"]}
        },
        {"POS": "VERB"},
        {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ", "VBP", "VBN"]}},
        {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ", "VBP", "VBN"]}},
        {"TAG": "RB", "OP": "*"},
        {"TAG": "VBN"}
        ],
          [
        {
           "DEP": {"IN": ["nsubj", "nsubjpass"]}
        },
        {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ", "VBP", "VBN"]}},
        {"POS": "VERB"},
        {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ", "VBP", "VBN"]}},
        {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ", "VBP", "VBN"]}},
        {"TAG": "RB", "OP": "*"},
        {"TAG": "VBN"}
        ],
          [
             {"DEP": "nsubjpass"},
             {"TAG": "VBG"},
             {"TAG": "VBN"}
          ],



        #5 mental verb with S (Attributes are now siblings)
        [
            {
                "TAG": {"IN": ["NNS", "NNPS"]}
            },
            {"LEMMA": "be", "TAG": {"IN":["VBD", "VBP", "VB"]}},
            {"LEMMA": {"IN":["come", "exist", "happen", "occur", "seem", "appear", "belong", "go", "arrive", "depart", "walk", "sleep", "die", "lie", "sit", "consist", "have", "contain", "resemble", "lack", "depend", "fit", "cost", "laugh", "cry", "become", "addict", "interest", "increase", "remain", "thrill", "feel", "learn", "prefer", "comprise", "join", "include", "enroll", "step", "install", "enter"]}, "TAG": "VBN"}
        ],

        [
            {
                "TAG": "PRP",
                "LOWER": {"IN": ["they", "we", "you"]}
            },
            {"LEMMA": "be", "TAG": {"IN":["VBD", "VBP", "VB"]}},
            {"LEMMA": {"IN":["come", "exist", "happen", "occur", "seem", "appear", "belong", "go", "arrive", "depart", "walk", "sleep", "die", "lie", "sit", "consist", "have", "contain", "resemble", "lack", "depend", "fit", "cost", "laugh", "cry", "become", "addict", "interest", "increase", "remain", "thrill", "feel", "learn", "prefer"]}, "TAG": "VBN"}
        ],
        [
             {"LEMMA": "be", "TAG": {"IN":["VBD", "VBP"]}},
             {"DEP": "neg", "TAG": "RB", "OP": "?"},
             {"TAG": "RB", "OP": "*"},
              {"LEMMA": {"IN":["come", "exist", "happen", "occur", "seem", "appear", "belong", "go", "arrive", "depart", "walk", "sleep", "die", "lie", "sit", "consist", "have", "contain", "resemble", "lack", "depend", "fit", "cost", "laugh", "cry", "become", "addict", "interest", "increase", "remain", "thrill", "feel", "learn", "prefer", "join", "include", "enroll", "step", "install", "enter"]}, "TAG": "VBN"},
             {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct"]}, #object modifier
                "OP": "*"
            }

                     ],

        #singular mental verb with S
        [
            {
                "POS": "PRON",
                "TAG": "PRP"
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ"]}},
            {"LEMMA": {"IN":["come", "exist", "happen", "occur", "seem", "appear", "belong", "go", "arrive", "depart", "walk", "sleep", "die", "lie", "sit", "consist", "have", "contain", "resemble", "lack", "depend", "fit", "cost", "laugh", "cry", "become", "addict", "interest", "increase", "remain", "thrill", "feel", "learn", "study", "join", "include", "enroll", "step", "install", "enter"]}, "TAG": "VBN"}
        ],
        [
            {
                "POS": "PRON",
                "TAG": "PRP",
                "LOWER": {"IN": ["he", "she", "it"]}
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ"]}},
            {"LEMMA": {"IN":["come", "exist", "happen", "occur", "seem", "appear", "belong", "go", "arrive", "depart", "walk", "sleep", "die", "lie", "sit", "consist", "have", "contain", "resemble", "lack", "depend", "fit", "cost", "laugh", "cry", "become", "addict", "interest", "increase", "remain", "thrill", "feel", "learn", "study", "join", "include", "enroll", "step", "install", "enter"]}, "TAG": "VBN"}
        ],
        #be + mental verb VBN
        [
            {"LEMMA": "be", "TAG": {"IN":["VBD", "VBP", "VBZ"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {
              "LEMMA": {"IN":["come", "exist", "happen", "occur", "seem", "appear", "belong", "go", "arrive", "depart", "walk", "sleep", "die", "lie", "sit", "consist", "have", "contain", "resemble", "lack", "depend", "fit", "cost", "laugh", "cry", "become", "addict", "interest", "increase", "remain", "thrill", "feel", "learn", "freak out", "dress"]},
              "TAG": "VBN"
            }
        ],
        #6: to be VB
        [
            {"LOWER": "to", "POS": "PART"},
            {"LEMMA": "be", "POS": "AUX"},
            {"TAG": "VB"}
        ],
        #be + VB
        [

            {"LEMMA": "be", "TAG": {"IN": ["VB", "VBD", "VBZ", "VBP"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"TAG": "VB"}
        ],
        [
            {"TAG": "MD"},
            {"LEMMA": "be", "TAG": "VBN"},
            {"TAG": "VB", "POS": "VERB"}
        ],
        #8: Irreg passive candidate
        [
            {"LEMMA": "be", "TAG": {"IN": ["VB", "VBG", "VBD", "VBZ", "VBP"]}},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "RB", "OP": "*"},
            {
                "TAG": {"IN": ["VBD", "VBN"]},
                "TEXT": {"REGEX": ".+ed$"},
                "LEMMA": {"IN": [
                    "teach", "buy", "go", "take", "write", "eat",
                    "make", "come", "give", "find", "think", "bring",
                    "catch", "choose", "feel", "keep", "know", "leave",
                    "lose", "meet", "pay", "read", "run", "say", "sell",
                    "send", "sit", "stand", "tell", "wear", "throw", "win",
                    "bleed", "borne", "bite", "beat", "become", "begin", "bend", "bet", "blow", "break", "burst", "cost", "creep", "cut", "deal", "dig", "do", "dream", "draw", "drink", "drive", "eat", "fall", "feel", "feed", "fight", "find", "fly", "forget", "flee", "forbid", "forgive", "freeze", "get", "grow", "hang", "have", "grow", "hear", "hide", "hit", "hold", "hurt", "keep", "kneel", "lay", "lead", "lend", "let", "learn", "lie", "live", "light", "lose", "make", "mean", "pay", "put", "read", "ride", "ring", "rise", "rest", "run", "say", "see", "sell", "send", "set", "sew", "shake", "shoot", "sing", "sink", "sit", "sleep", "smell", "shine", "sink", "sit", "sleep", "smell", "show", "shut", "sing", "sink", "sit", "sleep", "smell", "speak", "show", "shut", "sing", "slide", "spend", "spit", "split", "spread", "stand", "steal", "stick", "sting", "stink", "strike", "swear", "sweep", "swim", "take", "teach", "tear", "tell", "think", "throw", "understand", "wake", "wear", "weep", "win"
                ]}
            }
        ],
        #9: be + VBN + dobj (no preposition BY)

        [
            {
                "TAG": "NNS",

                },
            {
                "LEMMA": "want",
                "TAG": "VBP"
            },
            {
                "LEMMA": "to",
                "TAG": "TO"
            },
            {
                "LEMMA": "be",
                "TAG": "VB"
            },
            {
                "TAG": "VBN",
                "POS": "VERB"
            },
            {
                "TAG": "PRP$", "OP": "?"
            },
            {
                "TAG": "NN",
                "DEP": "dobj"
            }

        ],
        [
            {
                "TAG": "NNP"
                },
            {"LEMMA": "be", "TAG": {"IN": ["VB", "VBG", "VBD", "VBZ", "VBP"]}},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "RB", "OP": "*"},
             {"TAG": {"IN": ["VBD", "VBN"]}},
           {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "oprd"
            },

        ],
        [

            {
                "TAG": {"IN": ["NNS", "NN"]}

            },
            {"LEMMA": "be", "TAG": {"IN": ["VB", "VBD", "VBZ", "VBP"]}},
            {
                "LEMMA": {"IN": ["not", "n't"]},
                "DEP": "neg",
                "OP": "?"
            },
            {"TAG": "RB", "OP": "*"},
            {"TAG": "VBN"},
            {"TAG": "DET", "OP": "?"},
            {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "dobj"
            }

        ],
        [
            {"DEP": "ROOT"},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "punct"]},
                "OP": "*"
            },
            {"TAG": "WP", "OP": "?"},
            {"LEMMA": "be", "TAG": {"IN": ["VB", "VBD", "VBZ", "VBP"]}},
            {
                "LEMMA": {"IN": ["not", "n't"]},
                "DEP": "neg",
                "OP": "?"
            },
            {"TAG": "RB", "OP": "*"},
            {"TAG": "VBN"},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "punct"]},
                "OP": "*"
            },
            {"DEP": "dobj"}

        ],
        [
            {"TAG": "PRP", "DEP": "nsubjpass"},
            {"LEMMA": "be", "TAG": {"IN": ["VBP", "VBD", "VBZ"]}},
            {"TAG": "VBN"},
            {
                "DEP": {"IN": ["dobj", "iobj", "pobj", "oprd"]}
            }
        ],

        #10 absence of "be" patterns
        [
            {"TAG": {"IN": ["NNS", "JJ"]}},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod", "nsubj"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["WDT", "WP"]},
                "OP": "?"
            },
            {
                "TAG": "VBN",
                "POS": "VERB"
            },
            {
                "DEP": "agent",
                "LOWER": "by",
                "OP": "?"
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "pobj",
                "OP": "?"
            }
        ],

        [
            {"TAG": "NNS"},
            {
                "TAG": "VBN",
                "POS": "VERB"
            },
            {"DEP": "agent", "LOWER": "by"},
            {"TAG": "PRP$", "OP": "?"},
            {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "pobj"
            }
        ],
        [
            {"TAG": "IN", "POS": "ADP"},
            {
                "TAG": "VBN",
                "POS": "VERB"
            }
        ],
        [
            {"TAG": "WDT"},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
        [
            {"TAG": "WP"},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
        [
            {
                "TAG": "NNS"
            },
            {
                "LEMMA": "HAVE",
                "POS": "AUX",
                "TAG": {"IN": ["VBP", "VBD"]}
            },
            {
                "TAG": "VBN",
                "POS": "VERB"
            },
            {
                "LEMMA": "BY",
                "DEP": "agent"
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {

                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "pobj"
            }

        ],
        [
            {
                "TAG": "PRP",
                 "LOWER": {"IN": ["they", "we", "you", "I"]}
            },
            {
                "LEMMA": "HAVE",
                "POS": "AUX",
                "TAG": {"IN": ["VBP", "VBD"]}
            },
            {
                "TAG": "VBN",
                "POS": "VERB"
            },
            {
                "LEMMA": "BY",
                "DEP": "agent"
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {

                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "pobj"
            }

        ],
        [

            {
                "TAG": "NNS",
                "OP": "?"
            },
           {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod", "pobj"]},
                "OP": "*"
            },
            {"TAG": "VBN"},
            {"TAG": "IN"},
           {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "pobj"
            }
        ],
        [
            {
                "TAG": "PRP",
                "OP": "?"
            },
           {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod", "pobj"]},
                "OP": "*"
            },
            {"TAG": "VBN"},
            {"TAG": "IN"},
           {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "pobj"
            }
        ],
        [

            {"TAG": "NNS"},
            {"TAG": {"IN": ["VB", "VBN", "VBP", "VBD", "VBZ"]}},
            {"TAG": "MD"}
        ],
        [

            {"TAG": {"IN": ["PRP", "NNS", "NN", "NNP", "NNPS"]}},
            {"TAG": "MD"},
             {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
        [

            {"TAG": "NNS"},
            {"TAG": {"IN": ["VB", "VBN", "VBP", "VBD", "VBZ"]}},
            {"TAG": "MD"}
        ],
        [
            {
              "TAG": {"IN": ["NN", "NNS", "NNP"]}

            },
            {"TAG": "VBN"},
            {"LEMMA": "BY", "DEP": "agent", "OP": "?"},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {
                "DEP": {"IN": ["dobj", "iobj", "pobj"]},
                 "OP": "*"
            }
        ],
        [
            {
                "TAG": {"IN": ["PRP", "NN"]}

            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
          {
                "TAG": {"IN": ["VBN", "VBD"]},
                "POS": "VERB"
            }
        ],
        [
            {
                "LEMMA": "it",
                "TAG": "PRP"
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {
                "TAG": "VBN",
                "POS": "VERB"
            }
        ],
    [

        {"DEP": "ROOT", "TAG": "JJ"},
        {"TAG": "VBN", "DEP": "acl"},
        {"TAG": "IN", "LEMMA": "as"},
        {
            "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "oprd", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct"]},
            "OP": "*"
        }
    ],

        [
            {"DEP": "csubjpass"},
             {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod", "dobj"]},
                "OP": "*"
        },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}

        ],
        [
            {"TAG": "MD"},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod", "dobj"]},
                "OP": "*"
        },
            {"TAG": "VBN"}
        ],



        #11 Rel clauses with wrong copula
        [
          {
                "TAG": {"IN": ["PRP", "NNS"]}
            },
            {"TAG": "WP", "OP": "?"},
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
         [
           {"TAG": {"IN": ["NNP", "NNS"]}},
          {
                "TAG": {"IN": ["PRP", "NNS"]}
            },
            {"TAG": "WP"},
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
        [
            {
                "TAG": {"IN":["NNS","NNPS"]}
            },
            {"TAG": "WP", "OP": "?"},
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
         [
            {"TAG": {"IN":["NN","NNP"]}},
            {"TAG": "WP", "OP": "?"},
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP", "VB"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
        [
            {
                "TAG": {"IN": ["PRP", "NNP", "NN"]}
            },
            {"TAG": "WP", "OP": "?"},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
        [
            {
                "TAG": "NNS"
            },
            {"TAG": "WP"},
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
        [
            {"TAG": {"IN": ["WP", "WDT"]}, "DEP": "nsubjpass"},
            {"LEMMA": "be", "TAG": {"IN":["VB", "VBP"]}, "DEP": "auxpass"},
            {"TAG": "VBN"}

        ],
        [
            {
                "TAG": {"IN": ["PRP", "NNS"]}
            },
            {"TAG": "WDT"},
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
        [
            {
                "TAG": "NNS"
            },
            {"TAG": "WDT"},
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
        [
            {
                "TAG": {"IN": ["PRP", "NNP"]}
            },
            {"TAG": "WDT"},
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
        [
            {
                "TAG": "NNS"
            },
            {"TAG": "WDT"},
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
        [
            {"TAG": {"IN": ["NN", "NNS", "PRP"]}},
            {"TAG": "RB", "OP":"?"},
            {
                "LEMMA": "BE",
                "TAG": {"IN": ["VB", "VBP"]}
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],
        [
            {"DEP": {"IN": ["nsubjpass", "nsubj", "csubjpass"]}},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"LEMMA": "be", "TAG": "VB"},
            {"TAG": "VBN"}
        ],
        [

            {
                "TAG": "NNS"
            },
            {"TAG": "WDT", "OP": "?"},
            {
                "POS": "AUX",
                "TAG": "VBP",
                "LEMMA": "do"
            },
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
           {"LEMMA": "be", "TAG": {"IN": ["VBN", "VB"]}},
            {"TAG": "VBN"}
        ],
        [
            {"LEMMA": "be", "TAG": {"IN": ["VB", "VBD", "VBP", "VBZ", "VBN"]}},
            {"LEMMA": "to", "TAG": "TO"},
            {"LEMMA": "be", "TAG": "VB"},
            {"TAG": "VBN"}
        ],
        [
            {"TAG": "DT", "LEMMA": "that"},
            {
                "TAG": {"IN": ["VBP", "VBD"]},
                "LEMMA": "be"
            },
            {"TAG": "VBN"}
        ],
         [
            {"TAG": "DT", "LEMMA": "that"},
            {
                "TAG": {"IN": ["VBP", "VBD"]},
               "LEMMA": "be"
            },
            {"TAG": "VBN"}
        ],
[
    {
      "TAG":"PRP", "LEMMA": {"IN": ["you", "they", "we"]}
    },
    {"TAG": "WP"},
    {"DEP": "relcl", "TAG": "VBZ"},
    {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod", "acomp", "auxpass"]},
                "OP": "*"
            },
    {"TAG": "VBN"}
],
[
    {
      "TAG":"PRP", "LEMMA": {"IN": ["you", "they", "we"]}
    },
    {"TAG": "WP"},
    {"DEP": "relcl", "TAG": "VBD", "TEXT": "was"},
    {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod", "acomp", "auxpass"]},
                "OP": "*"
            },
    {"TAG": "VBN"}
],
          [
    {
      "TAG":"PRP", "LEMMA": {"IN": ["she", "he", "it"]}
    },
    {"TAG": "WP"},
    {"DEP": "relcl", "TAG": "VBD", "TEXT": "were"},
    {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod", "acomp", "auxpass"]},
                "OP": "*"
            },
    {"TAG": "VBN"}
],
            [
    {
      "TAG":"PRP", "LEMMA": {"IN": ["she", "he", "it"]}
    },
    {"TAG": "WP"},
    {"DEP": "relcl", "TAG": "VBP", "TEXT": "are"},
    {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod", "acomp", "auxpass"]},
                "OP": "*"
            },
    {"TAG": "VBN"}
],
        #12 Supposed to be Active

        [
            {"TAG": "PRP", "LEMMA": "I"},
            {"TAG": {"IN": ["VBD", "VBP"]}, "LEMMA": "be"},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {
                "TAG": {"IN": ["VBN", "VB"]},
                "POS": "VERB",
                "LEMMA": {"IN":["come", "exist", "happen", "occur", "seem", "appear", "belong", "go", "arrive", "depart", "walk", "sleep", "die", "lie", "sit", "consist", "have", "contain", "resemble", "lack", "depend", "fit", "cost", "laugh", "cry", "become", "addict", "interest", "increase", "remain", "thrill", "feel", "learn", "study", "see", "fall", "anticipate", "ride", "try", "wait", "join", "include", "enroll", "step", "install", "enter"]}, "TAG": "VBN"},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det", "advmod", "prep",  "npadvmod", "punct", "advmod", "cc", "conj", "amod", "acomp", "auxpass"]},
                "OP": "*"
            },
            {"DEP": "pobj"}

        ],
        [
            {"TAG": "PRP"},
            {"TAG": {"IN": ["VBD", "VBP"]}, "LEMMA": "be"},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {
                "TAG": {"IN": ["VBN", "VB"]},
                "LEMMA": {"IN":["come", "exist", "happen", "occur", "seem", "appear", "belong", "go", "arrive", "depart", "walk", "sleep", "die", "lie", "sit", "consist", "have", "contain", "resemble", "lack", "depend", "fit", "cost", "laugh", "cry", "become", "addict", "interest", "increase", "remain", "thrill", "feel", "learn", "study", "see", "fall", "anticipate", "ride", "try", "diet", "join", "include", "enroll", "step", "install", "enter"]}
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det", "advmod", "prep",  "npadvmod", "punct", "advmod", "cc", "conj", "amod", "acomp", "auxpass"]},
                "OP": "*"
            },
            {"DEP": "pobj"}
        ],
         [
            {

                "TAG": {"IN": ["NNS", "NNPS", "PRP", "NN", "NNP"]}
            },
            {"LEMMA": "need", "TAG": "VBN"},
            {"LEMMA": "to", "TAG": "IN"},
           { "LEMMA": "BE", "TAG": "VB"},
           {"TAG": "VBN"},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"}
        ],
          [
              {"DEP": {"IN": ["nsubj", "nsubjpass"]}, "TAG": {"IN": ["NNS", "NNPS"]}},
              {
              "LEMMA": "be",
              "TAG": "VBZ"
              },
              {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
              {"TAG": "VBN", "DEP": "acomp"}
          ],
          [
              {"DEP": {"IN": ["nsubjpass", "nubj"]}},
              {"DEP": "auxpass"},
              {"TAG": "VBN", "LEMMA": {"IN": ["look", "get", "determine"]}},
              {"TAG": "RB", "OP": "?"},
              {"TAG": "JJ"}
          ],
          [
              {"DEP": "nsubjpass"},
              {"DEP": "auxpass", "POS": "AUX"},
              {"TAG": "VBN"},
               {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "relcl", "acl", "appos", "cc", "conj", "punct", "oprd", "case"]},
                "OP": "*"
            },
              {"DEP": {"IN": ["dobj", "attr", "nsubj"]}}
          ],

        #13 supposed to be PASSIVE
        [
            {"TAG": "PRP"},
            {"LEMMA": "HAVE", "POS": "AUX", "TAG": "VBN"},
            {"POS": "VERB", "TAG": "VBN"},
            {"TAG": "IN"},
            {
                "TAG": {"IN": ["NNS", "NN", "NNP"]},
                "DEP": "pobj"
            }
        ],
        [
            {"DEP": {"IN": ["nsubj", "nsubjpass"]}},
            {"LEMMA": "HAVE", "POS": "AUX"},
            {"POS": "VERB", "TAG": "VBN"},
            {"TAG": "IN", "LEMMA": "by", "DEP": "agent"},
             {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "oprd"]}, #object modifier
                "OP": "*"
            },
            {
                "TAG": {"IN": ["NNS", "NN", "NNP"]},
                "DEP": "pobj"
            }
        ],
        [
            {"TAG": "PRP", "LEMMA": "it"},
            {"POS": "VERB", "TAG": "VBD"},
            {"TAG": "DT", "DEP": "prep", "LEMMA": "as"},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "oprd"]}, #object modifier
                "OP": "*"
            }
        ],
         [
            {"TAG": "PRP", "LEMMA": "it"},
            {"POS": "VERB", "TAG": "VBD", "TEXT": "made"},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "oprd"]}, #object modifier
                "OP": "*"
            },
        ],
        [
            {

                "TAG": {"IN": ["NNP", "NN", "NNS", "NNPS"]},
            },
           {"TAG": {"IN": ["WDT", "WP"]}},
           {

                "TAG": {"IN": ["NNP", "NN", "NNS", "NNPS", "PRP"]},
            },
           {"DEP": "relcl"},
           {"TAG": "MD"},
           {"DEP": "ccomp"}

        ],
        [
            {

                "TAG": {"IN": ["NNP", "NN", "NNS", "NNPS", "PRP"]},
            },
            {"TAG": {"IN": ["NNP", "NN", "NNS", "NNPS", "PRP"]},
            },
           {"TAG": {"IN": ["VBZ", "VBP", "VB",]}},
          {"DEP": "agent", "LOWER": "by"},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "pobj"
            }

        ],

        #14 missing preposition after specific VBN: exposed, related, appealed without to, located without in, look without at
        [

            {
                "TAG": "NNS"
            },
           {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ", "VBP", "VB", "VBN"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {
                "LEMMA": {"IN": ["expose", "locate", "look", "relate", "appeal", "give", "send", "bring", "take", "show", "teach", "offer", "read", "write", "explain", "describe", "throw"]},
                "TAG": "VBN"},
            {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "dobj"
            }
        ],
        [
            {"TAG": "PRP$", "OP": "?"},
            {

                "TAG": {"IN": ["NNP", "NN"]}
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"LEMMA": "EXPOSE", "TAG": "VBN"},
            {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "dobj"
            }
        ],
        [
            {

                "TAG": {"IN": ["NNS", "NNPS"]}
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"LEMMA": "EXPOSE", "TAG": "VBN"},
            {"TAG": {"IN": ["NN", "NNS"]}, "DEP": "dobj"}
        ],
        [
            {"TAG": "PRP$", "OP": "?"},
            {

                "TAG": {"IN": ["NNS", "NNPS"]}
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"LEMMA": "EXPOSE", "TAG": "VBN"},
            {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "dobj"
            }
        ],

        #15 wrong preposition after specific VBN

          #"need to" instead of "need for"

        [
            {

                "TAG": {"IN": ["NNS", "NNPS"]}
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"LEMMA": "need", "TAG": "VBN"},
            {"LEMMA": "for", "TAG": "IN"},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {
                "DEP": {"IN": ["dobj", "iobj", "pobj"]},
                "OP": "*"
            }
        ],

          #16 prep + VBN (no "being before VBN")
        [
         {
             "TAG": "IN",
             "POS": "SCONJ"

          },
         {
             "TAG": "VBN",
             "POS": "VERB"
         }

        ],


        #20 get stress
        [
            {
                "POS": "VERB",
                "LEMMA": "get",
                "TAG": {"IN": ["VBD", "VBZ", "VBP"]},
                },
            {
                "TEXT": "stress",
                "LEMMA": "stress",
                "DEP": "dobj"
            }
        ],

        #get + NOT_IN VBN
         [
            {
                "POS": "VERB",
                "LEMMA": "get",
                "TAG": {"IN": ["VBD", "VBZ", "VBP"]},
                },
            {
                "TAG": {"IN": ["VB", "VBD", "VBP", "VBZ"]}}

        ],


        #21 WRONG VERB
          [
{"LEMMA": "have"},
{"TAG": "VBN"},
{"LOWER": "by"}
],
          [
{"LEMMA": "do"},
{"DEP": "neg", "OP": "?"},
{"LEMMA": "be"},
{"TAG": "VBN"}
],
          [
{"LEMMA": "be", "TAG": "VBD", "DEP": {"NOT_IN": ["auxpass"]}}],
        [
            {

                "TAG": "PRP",
                 "LOWER": {"IN": ["he", "she", "it"]}
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "NN", "LEMMA": "release"}
        ],
        [
            {

                "TAG": "PRP"
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": {"IN": ["VB", "VBZ", "VBP"]}}
        ],
        [
    {"TAG": {"IN": ["NNP", "NN"]}},
    {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ"]}},
    {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
    {"DEP": "neg", "TAG": "RB", "OP": "?"},
    {"TAG": {"IN": ["VB", "VBP", "VBD", "VBZ"]}}
],
        [
            {"TAG": "MD"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"LEMMA": "be", "TAG": "VB"},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"TAG": {"IN": ["VB", "VBP", "VBD", "VBZ"]}}
        ],
        [
            {"TAG": "MD"},
            {"LEMMA": "BE", "TAG": "VB"},
            {"TAG": {"IN": ["NN", "NNS", "NNP", "NNPS"]}},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod", "prep"]},
            },
            {"DEP": {"IN": ["dobj", "iobj", "pobj"]}},
            {"DEP": "agent", "LEMMA": "by"},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod", "prep"]},
            },
            {"DEP": {"IN": ["dobj", "iobj", "pobj"]}}
        ],
        [
            {

                "TAG": {"IN": ["NNP", "NN"]}
            },
           {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {
                "LEMMA": "get",
                "TAG": {"IN":["VBP", "VB"]}
            },
            {"TAG": "VBN"}
        ],
        [
            {

                "TAG": "PRP",
                "DEP": "nsubj"
            },
            {
                "LEMMA": "get",
                "TAG": {"IN":["VBP", "VB"]}
            },
            {"TAG": "NN",
             "TEXT": "stress",
             "DEP": "dobj"}
        ],
         [
            {

                "TAG": {"IN": ["NNS", "NNPS"]}
            },
           {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {
                "LEMMA": "get",
                "TAG": "VBZ"
            },
            {"TAG": "VBN"}
        ],
        [
            {

                "TAG": {"IN": ["NNS", "NNPS"]}
            },
           {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {
                "LEMMA": "get",
                "TAG": "VBZ"
            },
            {"TAG": "VBN"}
        ],
          [
            {"TAG": "VBN", "LEMMA": "suppose"},
          {"DEP": "aux", "LEMMA": "to"},
            {
                "TAG": {"IN": ["VBN", "VBG", "VBZ"]}
            }
        ],
          #wrong copula
        [
            {
                "TAG": "PRP",
                "DEP": "nsubjpass",
                "LOWER": {"IN": ["he", "she", "it"]}
            },
            {
                "LEMMA": "be",
                "TAG": "VBD",
                "TEXT": "were"
            },
            {
                "TAG": {"IN": ["RB", "RBS"]},
                "OP": "?"
            },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
            },
            {"TAG": "VBN", "DEP": "ROOT"}
        ],
          [
            {
                "TAG": "PRP",
                "DEP": "nsubjpass",
                "LOWER": {"IN": ["he", "she", "it"]}
            },
            {
                "LEMMA": "be",
                "TAG": "VBP",
                "TEXT": "are"
            },
            {
                "TAG": {"IN": ["RB", "RBS"]},
                "OP": "?"
            },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
            },
            {"TAG": "VBN", "DEP": "ROOT"}
        ],
          [

            {

                "TAG": {"IN": ["NNP", "NN"]}
            },
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "aux", "mark", "punct"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["RB", "RBR"]},
                "OP": "?"
            },
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "acomp"]},
                "OP": "*"
            },
             {
                "LEMMA": "be",
                "TAG": "VBP", "TEXT": "are"
             },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"},
           {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "acomp"]},
                "OP": "*"
            },
            {"TAG": "VBN"}
          ],
           [

            {

                "TAG": {"IN": ["NNP", "NN"]}
            },
             {
                "LEMMA": "be",
                "TAG": "VBP", "TEXT": "are"
            },
            {"TAG": "VBN"},
            {"DEP": "oprd"}
          ],
           [
            {

                "TAG": {"IN": ["NNP", "NN"]}
            },
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "aux", "mark", "punct"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["RB", "RBR"]},
                "OP": "?"
            },
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "acomp"]},
                "OP": "*"
            },
             {
                "LEMMA": "be",
                "TAG": "VBD", "TEXT": "were"
             },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"},
           {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "acomp"]},
                "OP": "*"
            },
            {"TAG": "VBN"}
           ],
          [
            {
                "TAG": "PRP",
                "LOWER": {"IN": ["they", "we", "you"]}
            },
            {
                "LEMMA": "be",
                "TAG": "VBD", "TEXT": "was"},
            {
                "TAG": {"IN": ["RB", "RBS"]},
                "OP": "?"
            },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
            },
            {"TAG": "VBN"}
        ],
          [
            {
                "TAG": "PRP",
                "LOWER": {"IN": ["they", "we", "you"]}
            },
            {
                "LEMMA": "be",
                "TAG": "VBZ", "TEXT": "is"},
            {
                "TAG": {"IN": ["RB", "RBS"]},
                "OP": "?"
            },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"
            },
            {"TAG": "VBN"}
        ],
          [
            {

                "TAG": {"IN": ["NNS", "NNPS"]}
            },
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "aux", "mark", "punct"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["RB", "RBR"]},
                "OP": "?"
            },
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "acomp"]},
                "OP": "*"
            },
             {
                "LEMMA": "be",
                "TAG": "VBD", "TEXT": "was"
             },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"},
           {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "acomp"]},
                "OP": "*"
            },
            {"TAG": "VBN"}
           ],
          [
            {

                "TAG": {"IN": ["NNS", "NNPS"]}
            },
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "aux", "mark", "punct"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["RB", "RBR"]},
                "OP": "?"
            },
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "acomp"]},
                "OP": "*"
            },
             {
                "LEMMA": "be",
                "TAG": "VBZ", "TEXT": "is"
             },
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"},
           {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss", "advmod", "npvadvmod", "prep", "pobj", "relcl", "acl", "appos", "cc", "conj", "punct", "acomp"]},
                "OP": "*"
            },
            {"TAG": "VBN"}
           ],

          #HAVE GET HURT
        [
            {
                "TAG": {"IN": ["NNP", "NNS", "NN", "PRP"]}, "DEP": {"IN": ["nsubj", "nsubjpass"]}

            },
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBP", "VB", "VBZ", "VBN"]},
            },
              {
    "TAG": {"IN": ["VBD", "VBP", "VBZ", "VB", "VBN"]},
    "TEXT": {"REGEX": "^(?!.*ed$).*"},
    "LEMMA": {"IN": [
                    "teach", "buy", "go", "take", "write", "see", "eat",
                    "make", "come", "give", "find", "think", "bring",
                    "catch", "choose", "feel", "keep", "know", "leave",
                    "lose", "meet", "pay", "read", "run", "say", "sell",
                    "send", "sit", "speak", "stand", "tell", "wear", "throw", "win", "hurt", "bleed", "borne"
                ]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"},
            {"TAG": "VBN"},
            {"DEP": "agent", "LOWER": "by"},
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {
                "TAG": {"IN": ["NN", "NNS"]},
                "DEP": "pobj"
            }
        ],
           [
            {
                "TAG": {"IN": ["NNP", "NNS", "NN", "PRP"]}, "DEP": {"IN": ["nsubj", "nsubjpass"]}

            },
            {
                "LEMMA": "be",
                "TAG": {"IN": ["VBD", "VBP", "VB", "VBZ", "VBN"]},
            },
              {
    "TAG": {"IN": ["VBD", "VBP", "VBZ", "VB", "VBN"]},
    "TEXT": {"REGEX": "^(?!.*ed$).*"},
    "LEMMA": {"IN": [
                    "teach", "buy", "go", "take", "write", "see", "eat",
                    "make", "come", "give", "find", "think", "bring",
                    "catch", "choose", "feel", "keep", "know", "leave",
                    "lose", "meet", "pay", "read", "run", "say", "sell",
                    "send", "sit", "speak", "stand", "tell", "wear", "throw", "win", "hurt", "bleed", "borne"
                ]}}
           ],
        [
            {

                "TAG": {"IN": ["NNS", "NNPS"]}
            },
           {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {
                "LEMMA": "be",
                "TAG": {"IN":["VBZ", "VB" "VBP", "VBD", "VBN"]}
            },
           {"TAG": "MD"},
           {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"}
        ],
        [
            {
                "TAG": "PRP",
                "LOWER": {"IN": ["he", "she", "it"]},
                "DEP": {"IN": ["nsubj", "nsubjpass"]}
            },
            {"TAG": {"IN": ["VBP", "VB"]}, "POS": "VERB"},
             {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
           {
                "LEMMA": "be",
                "TAG": "VBG",
                "DEP": "auxpass"
            },
            {
                "TAG": "VBN"
            }
        ],
          [
              {"DEP": "nsubj"},
              {"TAG": {"IN": ["VBP", "VB", "VBZ", "VBN", "VBD"]}, "POS": "VERB"},
              {"TAG": "IN", "LEMMA": "by"},
              {"DEP": "pobj"}
          ],
          [
              {"TAG": {"IN": ["VB", "VBP", "VBD", "VBZ"]}},
              {"TAG": "IN", "LEMMA": "by"},
              {"DEP": "pobj"}
          ],
          [
              {"TAG": "TO", "LEMMA": "to"},
              {"TAG": "VBN"}
              ],

       #22 VERB + BEING + VBN
        [
            {"DEP": "cc"},
             {

                "TAG": "PRP",
                "LOWER": {"IN": ["he", "she", "it"]}
            },
            {
                "POS": "VERB",
                "TAG": {"IN": ["VBP", "VB"]}
            },
            {
                "LEMMA": "be",
                "TAG": "VBG",
                "DEP": "auxpass"
            },
            {
                "TAG": "VBN",
                "DEP": {"IN":["xcomp", "advcl"]}
            }
        ],

 [
            {

                "TAG": "PRP",
                "LOWER": {"IN": ["they", "we", "you"]}
            },
            {
                "POS": "VERB",
                "TAG": {"IN": ["VBP", "VB"]}
            },
            {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "pre", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
            {
                "LEMMA": "be",
                "TAG": "VBG",
                "DEP": "auxpass"
            },
            {
                "TAG": "VBN",
                "DEP": {"IN":["xcomp", "advcl"]}
            }
        ],

        #23 wrong order
[
            {"TAG": "MD"},
            {"LEMMA": "be", "TAG": "VB"},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],

       [
            {"TAG": "MD"},
            {
                "DEP": "neg",
                "TAG": "RB",
                "OP": "?"},
            {"DEP": "nsubjpass"},
            {"LEMMA": "be", "TAG": "VB", "DEP": "auxpass"},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"TAG": "VBN"},
            {"TAG": "to", "DEP": "aux"},
            {"TAG": "VB"}
        ],
        # Inverted question: MD + neg + subject + be + VBN (wrong order)
[{"TAG": "MD"},
 {"DEP": "neg", "TAG": "RB", "OP": "?"},  # catches n't
 {"TAG": {"IN": ["PRP", "NNS", "NN", "NNP"]}},
 {"LEMMA": "be", "TAG": "VB"},
 {"TAG": "VBN", "DEP": {"NOT_IN": ["ROOT", "ccomp", "advcl", "xcomp"]}}
],

#no copula
[
    {"DEP": "nsubj"},
    {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det", "npadvmod", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
    {"TAG": {"IN": ["JJ", "VBN"]}},
    {"TAG": "IN", "LEMMA": "by"},
    {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det", "npadvmod", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
    {"DEP": "pobj"}
],
    [
    {"DEP": {"IN": ["nsubj", "nsubjpass"]}},
    {"POS": "AUX", "LEMMA": "do"},
    {"TAG": "RB", "DEP": "neg"},
               {"DEP": {"IN": ["amod", "compound", "poss", "nummod", "det", "npadvmod", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
    {"TAG": {"IN": ["JJ", "VBN"]}},
    {"TAG": "IN", "LEMMA": "by"},
    {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det", "npadvmod", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
    {"DEP": "pobj"}
],
           [
    {"DEP": {"IN": ["nsubj", "nsubjpass"]}},
    {"POS": "AUX", "LEMMA": "do"},
    {"TAG": "RB", "DEP": "neg"},
               {"DEP": {"IN": ["amod", "compound", "poss", "nummod", "det", "npadvmod", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            },
    {"LEMMA": "be", "TAG": "VB"},
    {"TAG": {"IN": ["JJ", "VBN"]}}
],
          [

    {"TAG": {"IN": ["NNS", "NN", "NNP", "NNPS", "PRP"]}},
    {"LEMMA": "have", "TAG": {"IN": ["VBP", "VBZ", "VBD"]}, "DEP": "aux"},
    {"LEMMA": {"NOT_IN": ["be"]}, "TAG": "VBN"},
    {"TAG": "VBG", "DEP": "xcomp"}
],
          #DOBJ IS FORBIDDEN

          [
    {"DEP": "nsubjpass"},
    {"LEMMA": "be", "DEP": "auxpass"},
    {"TAG": "VBN"},
    {"TAG": {"IN": ["RB", "RBR", "DET"]}, "OP": "*"},
    {"DEP": "dobj"}
],

          # Pattern to catch "is go", "am join", etc.
 [
    {"LEMMA": "be"},
    {"POS": "ADV", "OP": "*"},
    {"TAG": "VB"}
],
        # PRP(singular) + be + intransitive verb (any verb tag)
        [
            {"TAG": "PRP", "LOWER": {"IN": ["i", "he", "she", "it"]}},
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ", "VBP"]}},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {
                "LEMMA": {"IN": ["wait", "cry", "go", "come", "arrive", "depart",
                                  "walk", "run", "sleep", "sit", "lie", "die",
                                  "laugh", "emerge", "pop", "appear", "exist",
                                  "happen", "occur", "fall", "rise", "grow",
                                  "step", "belong", "remain"]},
                "TAG": {"IN": ["VBN", "VBD", "VB", "VBP", "VBZ"]}
            }
        ],

        # PRP(plural) + be + intransitive verb (any verb tag)
        [
            {"TAG": "PRP", "LOWER": {"IN": ["they", "we", "you"]}},
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {
                "LEMMA": {"IN": ["wait", "cry", "go", "come", "arrive", "depart",
                                  "walk", "run", "sleep", "sit", "lie", "die",
                                  "laugh", "emerge", "pop", "appear", "exist",
                                  "happen", "occur", "fall", "rise", "grow",
                                  "step", "belong", "remain"]},
                "TAG": {"IN": ["VBN", "VBD", "VB", "VBP", "VBZ"]}
            }
        ],

        # Noun + be + intransitive verb (any verb tag)
        [
            {"TAG": {"IN": ["NN", "NNP", "NNS", "NNPS"]}},
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ", "VBP"]}},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {
                "LEMMA": {"IN": ["wait", "cry", "go", "come", "arrive", "depart",
                                  "walk", "run", "sleep", "sit", "lie", "die",
                                  "laugh", "emerge", "pop", "appear", "exist",
                                  "happen", "occur", "fall", "rise", "grow",
                                  "step", "belong", "remain"]},
                "TAG": {"IN": ["VBN", "VBD", "VB", "VBP", "VBZ"]}
            }
        ],

        # Subject + be + VBN + retained object (broadened dep labels)
        [
            {"DEP": {"IN": ["nsubjpass", "nsubj"]}},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss",
                               "advmod", "npadvmod", "prep", "pobj",
                               "cc", "conj", "punct"]},
                "OP": "*"
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ", "VBP"]}},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"TAG": "VBN"},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss",
                               "advmod", "npadvmod", "cc", "conj", "punct"]},
                "OP": "*"
            },
            {"DEP": {"IN": ["dobj", "attr", "oprd", "nsubj"]}, "OP": "+"}
        ],

        # Simpler fallback — be + VBN + bare noun (no dep constraint)
        [
            {"DEP": {"IN": ["nsubjpass", "nsubj"]}},
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ", "VBP"]}},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {
                "TAG": "VBN",
                "LEMMA": {"NOT_IN": ["call", "name", "consider", "find", "make",
                                      "keep", "leave", "elect", "appoint", "prove",
                                      "show", "base", "regard", "deem", "judge",
                                      "declare", "think", "believe", "describe",
                                      "depict", "portray", "render", "allow",
                                      "send", "give", "teach", "tell", "show",
                                      "pay", "offer", "award", "grant", "lend"]}
            },
            {"TAG": {"IN": ["DT", "PRP$"]}, "OP": "?"},
            {"TAG": {"IN": ["NN", "NNS", "NNP", "NNPS"]}}
        ],

        # Modal + be + VBN + object (broadened)
        [
            {"TAG": "MD"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"LEMMA": "be", "TAG": "VB"},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"TAG": "VBN"},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss",
                               "advmod", "cc", "conj", "punct"]},
                "OP": "*"
            },
            {"DEP": {"IN": ["dobj", "attr", "oprd", "nsubj"]}}
        ],

        # to be + VBN + object
        [
            {"LOWER": "to", "TAG": "TO"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"LEMMA": "be", "TAG": "VB"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"TAG": "VBN"},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss",
                               "advmod", "cc", "conj", "punct"]},
                "OP": "*"
            },
            {"DEP": {"IN": ["dobj", "attr", "oprd", "nsubj"]}}
        ],

        # "has made by" — missing "been" in perfect passive
        [
            {"DEP": {"IN": ["nsubj", "nsubjpass"]}},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss",
                               "advmod", "cc", "conj", "punct"]},
                "OP": "*"
            },
            {"LEMMA": "have", "TAG": {"IN": ["VBZ", "VBD", "VBP"]}},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN", "LEMMA": {"NOT_IN": ["be"]}},
            {"DEP": "agent", "LOWER": "by"}
        ],

        # PRP/Noun + be + intransitive base form (was exist, is go)
        [
            {"TAG": {"IN": ["PRP", "NN", "NNP", "NNS", "NNPS"]}},
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ", "VBP"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {
                "LEMMA": {"IN": ["exist", "go", "come", "pop", "wait", "arrive",
                                  "depart", "walk", "emerge", "happen", "occur"]},
                "TAG": {"IN": ["VB", "VBP", "VBZ", "VBD"]}
            }
        ],

        # "was looked very lazy/pity" — be + look + adj
        [
            {"DEP": {"IN": ["nsubj", "nsubjpass"]}},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss",
                               "advmod", "cc", "conj", "punct"]},
                "OP": "*"
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ", "VBP"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"LEMMA": "look", "TAG": "VBN"},
            {"TAG": {"IN": ["RB", "RBS", "JJ", "JJR", "JJS"]}},
             {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            }
        ],
        [
            {"TAG": "PRP"},
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ", "VBP"]}},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"LEMMA": "look", "TAG": "VBN"},
            {"TAG": {"IN": ["RB", "RBS", "JJ", "JJR", "JJS"]}},
             {
                "DEP": {"IN": ["amod", "compound", "poss", "nummod", "det",  "npadvmod", "punct", "advmod", "cc", "conj", "amod"]},
                "OP": "*"
            }
        ],

        # "are comprised of"
        [
            {"DEP": {"IN": ["nsubj", "nsubjpass"]}},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss",
                               "advmod", "cc", "conj", "punct"]},
                "OP": "*"
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ", "VBP"]}},
            {"LEMMA": "comprise", "TAG": "VBN"},
            {"LOWER": "of", "TAG": "IN"}
        ],

        # "have get hurt" — wrong aux chain
        [
            {"LEMMA": "have", "TAG": {"IN": ["VBP", "VBZ", "VBD"]}},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"LEMMA": "get", "TAG": {"IN": ["VB", "VBP", "VBZ"]}},
            {"TAG": "VBN"}
        ],

        # Wrong past participle spelling (hurted, bited, etc.)
        [
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ", "VBP"]}},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {
                "TEXT": {"REGEX": ".+ed$"},
                "LEMMA": {"IN": ["hurt", "bite", "hit", "cut", "put", "let",
                                  "set", "split", "spread", "shut", "cost",
                                  "burst", "beat", "read", "lead", "bleed",
                                  "breed", "feed", "flee", "meet", "feel",
                                  "deal", "kneel", "sleep"]},
                "TAG": {"IN": ["VBN", "VBD"]}
            }
        ],
        # Also catch by TEXT directly for common misspellings
        [
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ", "VBP"]}},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TEXT": {"IN": ["hurted", "bited", "hitted", "costed", "bursted",
                             "beated", "feeled", "dealed", "readed", "leaded",
                             "meeted", "cuted", "putted", "letted", "setted"]}}
        ],

        # "are talked about" — talk is intransitive
        [
            {"TAG": "PRP"},
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ", "VBP"]}},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"LEMMA": "talk", "TAG": "VBN"},
            {"LOWER": "about", "TAG": "IN"}
        ],
        [
            {"DEP": {"IN": ["nsubj", "nsubjpass"]}},
            {
                "DEP": {"IN": ["det", "amod", "compound", "nummod", "poss",
                               "advmod", "cc", "conj", "punct"]},
                "OP": "*"
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ", "VBP"]}},
            {"LEMMA": "talk", "TAG": "VBN"},
            {"LOWER": "about", "TAG": "IN"}
        ],

        # I + be + VBN(join/include) — agent wrongly as patient
        [
            {"TAG": "PRP", "LOWER": "i", "DEP": {"IN": ["nsubj", "nsubjpass"]}},
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBP"]}},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {
                "LEMMA": {"IN": ["join", "include", "enroll", "register",
                                  "enter", "step"]},
                "TAG": {"IN": ["VBN", "VBD"]}
            }
        ],

        # "to be not VBN" — negation in wrong position
        [
            {"LOWER": "to", "TAG": "TO"},
            {"LEMMA": "be", "TAG": "VB"},
            {"DEP": "neg", "TAG": "RB"},
            {"TAG": "VBN"}
        ],

        # "is felt pained" — feel passive + complement
        [
            {"TAG": "PRP"},
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ", "VBP"]}},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"LEMMA": "feel", "TAG": "VBN"},
            {"TAG": {"IN": ["JJ", "VBN", "NN"]}}
        ],

        # NNS + VBZ(is) + VBN — plural noun + singular be
        # "students is very enjoyed"
        [
            {"TAG": "NNS", "DEP": {"IN": ["nsubj", "nsubjpass"]}},
            {"LEMMA": "be", "TAG": "VBZ", "TEXT": "is"},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"}
        ],

        # NN(singular) + VBP(are) + VBN — singular noun + plural be
        # "my privacy are opened", "these talk are considered"
        [
            {"TAG": {"IN": ["DT", "PRP$"]}, "OP": "?"},
            {"TAG": "NN", "DEP": {"IN": ["nsubj", "nsubjpass"]}},
            {"LEMMA": "be", "TAG": "VBP", "TEXT": "are"},
            {"TAG": {"IN": ["RB", "RBS"]}, "OP": "?"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"TAG": "VBN"},
            {"DEP": "oprd", "OP": "?"}
        ],

        # "was exist" — specific catch
        [
            {"TAG": {"IN": ["NN", "NNP", "NNS", "NNPS", "PRP"]}},
            {"LEMMA": "be", "TAG": "VBD"},
            {"LEMMA": "exist", "TAG": {"IN": ["VB", "VBZ", "VBP", "VBD", "VBN"]}}
        ],

        # "was determined [adjective]" — determine + adj complement
        [
            {"DEP": {"IN": ["nsubj", "nsubjpass"]}},
            {
                "DEP": {"IN": ["det", "amod", "compound", "poss", "cc", "conj"]},
                "OP": "*"
            },
            {"LEMMA": "be", "TAG": {"IN": ["VBD", "VBZ", "VBP"]}},
            {"LEMMA": "determine", "TAG": "VBN"},
            {"TAG": {"IN": ["RBR", "RBS", "RB", "JJR", "JJS", "JJ"]}}
        ],

        # "it is not formed [food]" — form + product noun
        [
            {"TAG": "PRP", "LOWER": "it"},
            {"LEMMA": "be", "TAG": "VBZ"},
            {"DEP": "neg", "TAG": "RB", "OP": "?"},
            {"LEMMA": "form", "TAG": "VBN"},
            {"TAG": {"IN": ["JJ", "RB"]}, "OP": "?"},
            {"TAG": {"IN": ["NN", "NNS"]}}
        ],
          [
    {"DEP": "aux", "LEMMA": "have"},
    {"LEMMA": "get", "TAG": "VB", "DEP": "auxpass"},
    {"TAG": "VBN"}
],
          [
              {"DEP": "PUNCT"},
              {"TAG": "VB", "LEMMA": "be"},
              {"TAG": "VBN"},
              {"TAG": "JJ"}

          ]

        # ══════════════════════════════════════════════════════════════════
        # END NEW RULES v2
        # ══════════════════════════════════════════════════════════════════
    ]

    # Assuming this is inside a function like: def check_passive_errors(doc, patterns, verbose=False):

    matcher = Matcher(nlp.vocab)
    matcher.add("ErrorPassiveVoice", patterns, greedy="LONGEST")
    matches = matcher(doc, as_spans=True)
    matches = spacy.util.filter_spans(matches)

    # adjectives that look like passive but are actually adjectival
    excluded_adj = {
        "related", "located", "known", "based", "used", "shown", "given",
        "moved", "discouraged", "expected", "supposed", "forced", "recognized",
        "formed", "departed", "arrived", "gone", "thrilled", "interested"
    }

    # verbs that can legitimately take objects in passive contexts
    SAFE_DITRANSITIVES = {"give", "show", "tell", "send", "offer", "teach"}

    filtered = []

    for span in matches:

        # remove adjectival VBN
        if span.root.lemma_ in excluded_adj:
            continue

        # find the VBN inside the span (important for passive)
        main_verb = None
        for token in span:
            if token.tag_ == "VBN":
                main_verb = token
                break

        # check if verb has direct object
        has_dobj = False
        if main_verb:
            has_dobj = any(t.dep_ == "dobj" for t in main_verb.children)

        # filter safe ditransitives
        if has_dobj and main_verb and main_verb.lemma_ in SAFE_DITRANSITIVES:
            continue

        filtered.append(span)

    # ignore if correct passive exists
    if correct(new_text) > 0:
        if verbose:
            print("Correct passive detected, ignoring error.")
        return 0

    # imperative filter: "Do not be discouraged"
    has_subject = any(t.dep_ in ["nsubj", "nsubjpass"] for t in doc)
    has_do_not_be = any(t.lemma_ == "do" and t.tag_ == "VB" for t in doc)

    if has_do_not_be and not has_subject:
        return 0

    if verbose:
        print("Error passive voice usages detected:")
        for match in filtered:
            print(match)

    return 1 if len(filtered) > 0 else 0
print('✅ correct() and error() functions defined.')


In [ ]:
print(df.columns.tolist())
print(df.head(5))

In [ ]:
nlp = spacy.load('en_core_web_sm')

def classify_sentence(text):
    if not isinstance(text, str) or text.strip() == '':
        return 0, 0

    try:
        c = correct(text)
    except Exception:
        c = 0

    try:
        e = error(text) if c == 0 else 0
    except Exception:
        e = 0

    return c, e

results = df['text'].apply(classify_sentence)
df['correct'] = results.apply(lambda x: x[0])
df['error']   = results.apply(lambda x: x[1])

print('✅ Done.')
print(f'   Total sentences processed : {len(df)}')
print(f'   Correct passive : {(df["correct"] > 0).sum()} sentences')
print(f'   Error passive   : {(df["error"] > 0).sum()} sentences')
print(f'   No passive      : {((df["correct"] == 0) & (df["error"] == 0)).sum()} sentences')
print()
display(df[['student_number', 'text', 'correct', 'error']].head(15))

In [ ]:
df['c_correct'] = df['new_text'].apply(correct)
df['c_error']   = df['new_text'].apply(error)

# Binarize human labels to 0 or 1 for metric calculation
df['actual_correct']   = (df['correct'] > 0).astype(int)
df['actual_error']     = (df['error'] > 0).astype(int)

# Binarize system predictions to 0 or 1 for metric calculation
df['predicted_correct'] = (df['c_correct'] > 0).astype(int)
df['predicted_error']   = (df['c_error'] > 0).astype(int)

print('✅ Predictions calculated and columns added to DataFrame.')
display(df.head())

---
## STEP 9: Save Full Results to CSV


In [ ]:
output_cols = ['student_number', 'text', 'c_correct', 'predicted_correct', 'c_error', 'predicted_error']
df[output_cols].to_csv('corpus_results.csv', index=False)
print('✅ Saved to corpus_results.csv')
display(df[output_cols].head(20))

In [ ]:
# ── Rows where human count ≠ system count (CORRECT passive) ────────────

print('═' * 70)
print('CORRECT PASSIVE — rows where human count ≠ system count')
print('═' * 70)
diff_correct = df[df['correct'] != df['c_correct']][
    ['student_number', 'text', 'correct', 'c_correct', 'correct_class']
].copy()
diff_correct.columns = ['student_number', 'text', 'human_correct', 'system_correct', 'classification']
print(f'Total rows with different counts: {len(diff_correct)}')
print()
display(diff_correct)

print()
print('─' * 70)
print('Breakdown:')
print(f'  System overcounted  (FP) : {(diff_correct["classification"] == "FP").sum()} rows')
print(f'  System undercounted (FN) : {(diff_correct["classification"] == "FN").sum()} rows')
print(f'  (Note: TP/TN rows are excluded — those counts matched)')


# ── Rows where human count ≠ system count (ERROR passive) ──────────────

print()
print()
print('═' * 70)
print('ERROR PASSIVE — rows where human count ≠ system count')
print('═' * 70)
diff_error = df[df['error'] != df['c_error']][
    ['student_number', 'text', 'error', 'c_error', 'error_class']
].copy()
diff_error.columns = ['student_number', 'text', 'human_error', 'system_error', 'classification']
print(f'Total rows with different counts: {len(diff_error)}')
print()
display(diff_error)

print()
print('─' * 70)
print('Breakdown:')
print(f'  System overcounted  (FP) : {(diff_error["classification"] == "FP").sum()} rows')
print(f'  System undercounted (FN) : {(diff_error["classification"] == "FN").sum()} rows')
print(f'  (Note: TP/TN rows are excluded — those counts matched)')


# ── Summary ─────────────────────────────────────────────────────────────

print()
print()
print('═' * 70)
print('SUMMARY')
print('═' * 70)
print(f'  CORRECT passive mismatches : {len(diff_correct)} rows')
print(f'  ERROR passive mismatches   : {len(diff_error)} rows')
total_any = df[(df['correct'] != df['c_correct']) | (df['error'] != df['c_error'])]
print(f'  Rows with ANY mismatch     : {len(total_any)} rows out of {len(df)} total')
print(f'  Rows perfectly matched     : {len(df) - len(total_any)} rows out of {len(df)} total')


# ── Save to CSV ──────────────────────────────────────────────────────────

diff_correct.to_csv('mismatch_correct.csv', index=False)
diff_error.to_csv('mismatch_error.csv', index=False)
total_any[['student_number', 'text',
           'correct', 'c_correct', 'correct_class',
           'error',   'c_error',   'error_class']].to_csv('mismatch_all.csv', index=False)

print()
print('✅ Files saved:')
print('   mismatch_correct.csv  — correct passive mismatches only')
print('   mismatch_error.csv    — error passive mismatches only')
print('   mismatch_all.csv      — all mismatches combined')


# ── Download in Colab ────────────────────────────────────────────────────

from google.colab import files
files.download('mismatch_correct.csv')
files.download('mismatch_error.csv')
files.download('mismatch_all.csv')
